# 1. Load packages

In [ ]:
import scipy.sparse as sp
import pandas as pd
import scanpy as sc
import pandas as pd
import igraph as ig<
import matplotlib.pyplot as plt
import networkx as nx
from mygene import MyGeneInfo

In [ ]:
import numpy as np
np.NAN = np.nan

In [ ]:
import omnipath as op

In [ ]:
from markov_clustering import run_mcl, get_clusters

In [ ]:
from matplotlib.colors import rgb2hex

In [ ]:
import leidenalg

In [ ]:
from omnipath.interactions import OmniPath

# 2. load data

In [ ]:
adata = sc.read("/storage/users/data/PANC/H5AD_file/adata_expr_mean_by_group.h5ad")

In [ ]:
adata

In [ ]:
adata_full = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D.h5ad')

In [ ]:
adata_full

# 3. Get edges

## 3.1.  Download OmniPath interactions

In [ ]:


# 1. See what parameters and resources are available
print("Query params:", OmniPath.params())        # dict of valid query-args :contentReference[oaicite:0]{index=0}
print("All resources:", OmniPath.resources())    # tuple of resource names :contentReference[oaicite:1]{index=1}

# 2. Download the full curated OmniPath network
inter = OmniPath.get()                          
print(f"Raw OmniPath: {inter.shape[0]:,} interactions")   # DataFrame



In [ ]:
# --- Define which resources to focus on (if desired) ---
focus_resources = [
    # Core signaling
    "KEGG", "Reactome", "SIGNOR",
    # Gene regulons
    "DoRothEA", "TRRUST",
    # PPI scaffold
    "BioGRID", "IntAct",
    # PTM detail (optional)
    "PhosphoSite", "iPTMnet",
    # Cell–cell communication
    "CellPhoneDB",
    # Legacy / additional
    "UniProt_LRdb", "HPRD", "Adhesome"
]

# --- Toggle filtering ---
use_mask = False  # Set to False to skip filtering by source

# --- Apply mask if requested ---
if use_mask:
    mask = inter["sources"].str.split(";").apply(
        lambda srcs: any(r in srcs for r in focus_resources)
    )
    inter = inter.loc[mask].copy()
    print(f"After resource filter: {inter.shape[0]:,} interactions")
else:
    print(f"Skipping resource filter: using all {inter.shape[0]:,} interactions")


In [ ]:
inter

## 3.2.  Quality filters  (mirror your R code)

In [ ]:
import numpy as np

# 1. Copy from the correct DataFrame (not the built-in iter)
df = inter.copy()

# 2. Base QC: curation effort & consensus direction
df = df[df["curation_effort"] >= 3]
#df = df[df["consensus_direction"] == 2]

# 3. Assign signed “type” and drop ambiguous edges
df["type"] = np.nan
df.loc[
    df["is_stimulation"].astype(bool) | df["consensus_stimulation"].astype(bool),
    "type"
] = "activation"
df.loc[
    df["is_inhibition"].astype(bool)  | df["consensus_inhibition"].astype(bool),
    "type"
] = "inhibition"
df = df.dropna(subset=["type"])

# 4. Publication support: ≥2 distinct papers
df = df[df["n_references"] >= 3]

# 5. Cross-resource consensus: seen in ≥2 resources
df = df[df["n_sources"] >= 1]

print(f"Rows after HQ filtering: {df.shape[0]:,}")
df.head()


## 3.3.  Quick glance at interaction types

In [ ]:
type_counts = df["type"].value_counts()
ax = type_counts.plot(kind="bar")
ax.set_ylabel("# interactions")
ax.set_title("Signed interactions in filtered network")
plt.tight_layout()
plt.show()

## 3.4.  Map IDs: ENSEMBL ↔ UniProt and Gene Symbol

In [ ]:
adata

In [ ]:
adata.X

In [ ]:
row_names = adata.obs_names.tolist()
col_names = adata.var_names.tolist()
print(f"{len(row_names)} rows:", row_names[:5])
print(f"{len(col_names)} cols:", col_names[:5])

In [ ]:
# 1. Build expr_mean DataFrame: rows=ENSG, cols=leiden groups
expr_mean = pd.DataFrame(
    adata.layers["mean"].T,
    index=adata.var_names,    # ENSG IDs
    columns=adata.obs_names    # clusters
)

# 2. Query MyGeneInfo for Swiss-Prot IDs
mg = MyGeneInfo()
results = mg.querymany(
    expr_mean.index.tolist(),
    scopes="ensembl.gene",
    fields="uniprot.Swiss-Prot",
    species="human",
    returnall=False
)

In [ ]:
# 3. Build ENSG→UniProt map (take first canonical isoform)
ensg2up = {}
for hit in results:
    ens = hit.get("query")
    if hit.get("notfound"):
        continue
    up = hit.get("uniprot", {}).get("Swiss-Prot")
    if isinstance(up, list):
        ensg2up[ens] = up[0]
    elif isinstance(up, str):
        ensg2up[ens] = up

# 4. Add UniProtID and gene_symbol columns to expr_mean
#    – gene_symbol comes from adata_full.var["gene_symbol"]
symbol_map = adata_full.var.set_index("ensembl_gene_id")["gene_symbol"].to_dict()

expr_mean["UniProtID"]   = expr_mean.index.map(ensg2up)
expr_mean["gene_symbol"] = expr_mean.index.map(symbol_map)

# 5. Drop any rows that failed to map either ID or symbol
expr_mean = expr_mean.dropna(subset=["UniProtID", "gene_symbol"])

# 6. (Optional) If you want one row per protein rather than ENSG, you can collapse:
# expr_mean = expr_mean.groupby(["UniProtID","gene_symbol"]).mean()

# 7. Check
print(f"Annotated expr_mean shape: {expr_mean.shape}")
print(expr_mean[["UniProtID","gene_symbol"]].head())

In [ ]:
expr_mean

In [ ]:
print(expr_mean.columns.tolist())

# 4. Get Nodes: get filtered expression matrix but per cell/barcode expression (not the mean)

In [ ]:
import pandas as pd
from scipy import sparse

# 1. Build ENSG→UniProt map (as before)
ensg2up = {}
for hit in results:
    ens = hit.get("query")
    if hit.get("notfound"):
        continue
    up = hit.get("uniprot", {}).get("Swiss-Prot")
    if isinstance(up, list):
        ensg2up[ens] = up[0]
    elif isinstance(up, str):
        ensg2up[ens] = up

# 2. Build ENSG→gene_symbol map (as before)
symbol_map = adata_full.var.set_index("ensembl_gene_id")["gene_symbol"].to_dict()

# 3. Extract raw expression matrix and transpose to (genes × cells)
X = adata_full.X
if sparse.issparse(X):
    mat = X.toarray()         # ← use .toarray() instead of .A
else:
    mat = X.copy()            # if it’s already a dense array

mat_T = mat.T  # now shape is (n_genes, n_cells)

expr_cells = pd.DataFrame(
    mat_T,
    index=adata_full.var.index,    # Ensembl IDs
    columns=adata_full.obs_names   # cell barcodes
)

# 4. Annotate with UniProtID and gene_symbol
expr_cells["UniProtID"]   = expr_cells.index.map(ensg2up)
expr_cells["gene_symbol"] = expr_cells.index.map(symbol_map)

# 5. Drop any rows with missing annotations
expr_cells = expr_cells.dropna(subset=["UniProtID", "gene_symbol"])

# 6. (Optional) Reorder so that identifiers come first
cols_order = ["UniProtID", "gene_symbol"] + [
    c for c in expr_cells.columns if c not in {"UniProtID", "gene_symbol"}
]
expr_cells = expr_cells[cols_order]

# 7. Quick sanity check
print(f"Annotated expr_cells shape: {expr_cells.shape}")
print(expr_cells[["UniProtID", "gene_symbol"]].head())


# 5. Build the network (nodes: expression; edges: omnipath) and get the biggest fuly connected componente (GCC)

## 5.1 Build the network

In [ ]:
# ─── Build the set of UniProt IDs actually in your filtered expr_mean ─────────
gene_set = set(expr_mean["UniProtID"].dropna().unique())
print(f"{len(gene_set):,} UniProt IDs in the expression matrix")

# ─── Now filter your OmniPath interactions df by that set ────────────────────
df_expr = df[
    df["source"].isin(gene_set) &
    df["target"].isin(gene_set)
].copy()
print(f"Edges after expression filter: {df_expr.shape[0]:,}")

# ─── And continue annotating symbols and building the igraph ─────────────────
symbol_map = dict(zip(
    expr_mean["UniProtID"],
    expr_mean["gene_symbol"]
))

df_expr["source_symbol"] = df_expr["source"].map(symbol_map)
df_expr["target_symbol"] = df_expr["target"].map(symbol_map)

import igraph as ig
edge_tuples = df_expr[["source","target","type"]].itertuples(index=False, name=None)
g_full = ig.Graph.TupleList(edge_tuples, directed=True, edge_attrs=["type"])
g_full.vs["gene_symbol"] = [symbol_map.get(name, "") for name in g_full.vs["name"]]

print(f"Global graph – nodes: {g_full.vcount()}, edges: {g_full.ecount()}")


In [ ]:
# 1. Filter to edges where both source & target were measured
df_expr = df[
    df["source"].isin(gene_set) &
    df["target"].isin(gene_set)
].copy()
print(f"Edges after expression filter: {df_expr.shape[0]:,}")

# 2. Build a mapping from UniProt ID → gene_symbol
#    (expr_mean has columns “UniProtID” and “gene_symbol”)
symbol_map = dict(zip(
    expr_mean["UniProtID"],
    expr_mean["gene_symbol"]
))

# 3. Annotate edges with source/target symbols
df_expr["source_symbol"] = df_expr["source"].map(symbol_map)
df_expr["target_symbol"] = df_expr["target"].map(symbol_map)

# 4. Build directed igraph using TupleList (type as edge attribute)

edge_tuples = df_expr[["source","target","type"]].itertuples(index=False, name=None)
g_full = ig.Graph.TupleList(
    edge_tuples,
    directed=True,
    edge_attrs=["type"]
)
print(f"Global graph – nodes: {g_full.vcount()}, edges: {g_full.ecount()}")

# 5. Annotate each vertex with its gene symbol
g_full.vs["gene_symbol"] = [ symbol_map.get(v["name"], "") for v in g_full.vs ]

# 6. (Optional) Verify a few nodes
for v in g_full.vs[:15]:
    print(f"{v['name']} → {v['gene_symbol']}")

## 5.2 Check for genes of interest

In [ ]:
missing_symbols = df_expr[
    df_expr["source_symbol"].isna() | df_expr["target_symbol"].isna()
]
print(f"Edges with missing symbols: {missing_symbols.shape[0]}")


In [ ]:
genes_of_interest = [
    "PLAU", "TGFB", "CDK1", "AURKA", "RRM2", "MYBL2", "DTL", "MELK",
    "CENPF", "HELLS", "CENPK", "FN1", "MBOAT2", "S100A2", "PGM2L1",
    "TPX2", "BIRC5"
]

# Extend with the new symbols (filter out Nones)
genes_of_interest.extend([g_full for g_full in new_symbols if g_full is not None])
genes_of_interest = list(set(genes_of_interest))  # deduplicate
genes_of_interest

In [ ]:
new_symbols = [symbol_map.get(uid, None) for uid in uniprot_ids]
print(list(zip(uniprot_ids, new_symbols)))

In [ ]:
# Build a mapping of UniProtID → gene_symbol for all nodes in the graph
node_symbols = { v["name"]: v["gene_symbol"] for v in g_full.vs }

# Filter to nodes whose symbol matches your list
present_nodes = [v for v in g_full.vs if v["gene_symbol"] in genes_of_interest]

print("Present genes in network:")
for v in present_nodes:
    print(f"{v['gene_symbol']} (ID: {v['name']})")

In [ ]:
for v in present_nodes:
    deg_in = g_full.degree(v, mode="in")
    deg_out = g_full.degree(v, mode="out")
    neighbors = [g_full.vs[n]["gene_symbol"] for n in g_full.neighbors(v)]
    print(f"{v['gene_symbol']} – in-degree: {deg_in}, out-degree: {deg_out}, neighbors: {neighbors}")

In [ ]:
# --- Build mapping from UniProt ID → gene symbol
node_symbols = {v["name"]: v["gene_symbol"] for v in g_full.vs}

# --- Filter to vertices of interest by gene symbol
present_nodes = [v for v in g_full.vs if v["gene_symbol"] in genes_of_interest]

# --- Precompute betweenness centrality (optional: set directed=True or False)
print("Calculating betweenness centrality (this may take a few seconds)...")
betweenness = g_full.betweenness(directed=True)
# Indexed by vertex ID, not name — so we access with v.index

# --- Report metrics for each gene of interest
for v in present_nodes:
    gene = v["gene_symbol"]
    uid = v["name"]  # UniProt
    deg_in = g_full.degree(v, mode="in")
    deg_out = g_full.degree(v, mode="out")
    btwn = betweenness[v.index]
    neighbors = [g_full.vs[n]["gene_symbol"] for n in g_full.neighbors(v)]
    
    print(f"🔹 {gene} ({uid})")
    print(f"   - In-degree:     {deg_in}")
    print(f"   - Out-degree:    {deg_out}")
    print(f"   - Betweenness:   {btwn:.2f}")
    print(f"   - Neighbors:     {neighbors}")


## 5.3 Get the vertex attributes

In [ ]:
import pandas as pd

# --- 1. List available attributes -----------------------
print("Vertex attributes:", g_full.vs.attribute_names())
print("Edge attributes:  ", g_full.es.attribute_names())

# --- 2. Pull out a DataFrame of vertices ---------------
vertices = pd.DataFrame({
    "UniProtID":    g_full.vs["name"],
    "gene_symbol":  g_full.vs["gene_symbol"],
})
print(vertices.head())

# --- 3. Pull out a DataFrame of edges ------------------
# Note: igraph edges don’t automatically carry source/target names in a column,
#       so we reconstruct them via the endpoints of each edge.
edge_list = []
for e in g_full.es:
    src, tgt = g_full.vs[e.tuple[0]]["name"], g_full.vs[e.tuple[1]]["name"]
    edge_list.append({
        "source": src,
        "target": tgt,
        "type":   e["type"]
    })
edges = pd.DataFrame(edge_list)
print(edges.head())

## 5.4 Build a new gcc from g_full

In [ ]:
import igraph as ig

# 2) Inspect available attributes (optional)
print("Vertex (node) attributes:", g_full.vs.attributes())
print("Edge attributes:", g_full.es.attributes())

# 3) Compute components (weakly for directed graphs; same for undirected)
mode = "weak"  # use "strong" if you specifically want strongly connected components
components = g_full.components(mode=mode)

# 4) Extract the largest component
largest_component = components.giant()
print(f"Full graph: {g_full.vcount()} nodes, {g_full.ecount()} edges")
print(f"Largest component: {largest_component.vcount()} nodes, {largest_component.ecount()} edges")

# 5) Use largest component for further calculations
g = largest_component

# 6) Genes-of-interest presence in GCC (prefers 'gene_symbol', falls back to 'name')
attr = "gene_symbol" if "gene_symbol" in g.vs.attributes() else "name"
gcc_node_names = set(g.vs[attr])

present_in_gcc = [gene for gene in genes_of_interest if gene in gcc_node_names]
print(f"Genes of interest in GCC ({attr}): {present_in_gcc}")


In [ ]:
# Extract weakly connected giant component from your graph `g`

## g has uniprot DI
## from a new g from gcc
## get largets component network
## Scale the expression values (expr_mean) rowwise
## get expression values from expr_mean for the largest componente (tranlate Ids). 
## from a hub network (reduce the network (number of node and edges) for pancy
## export smaller network hubnetwork and the expression matrix for Pancy
## run the following code with g or gcc
### interesting are nodes, edges, density over time 
### run the 

In [ ]:
g

## 5.5 Run betweenness centrality only on the GCC

In [ ]:
# Compute betweenness centrality on the largest component
btwn = largest_component.betweenness(directed=True)

# Collect centrality and degree stats
central_nodes = [
    {
        "gene_symbol": v["gene_symbol"],
        "uniprot_id": v["name"],
        "betweenness": btwn[v.index],
        "in_degree": largest_component.degree(v, mode="in"),
        "out_degree": largest_component.degree(v, mode="out")
    }
    for v in largest_component.vs
]

# Make into a DataFrame
df_gcc_stats = pd.DataFrame(central_nodes)

# Sort by betweenness
df_gcc_stats.sort_values("betweenness", ascending=False, inplace=True)

# Show top 10
top_central = df_gcc_stats.head(40)
print(top_central)


In [ ]:
g

# 6. Subset networks state wise: Build per-group/state subnetworks from GCC (expression-filtered)

In [ ]:
expr_mean

## 6.1 Generate Group/State‐Specific (e.g. time-bin) Subnetworks Based on Expression

In [ ]:
import igraph as ig

# Prepare a mapping UniProtID → gene_symbol
symbol_map = dict(zip(expr_mean["UniProtID"], expr_mean["gene_symbol"]))

# Parameters
expression_threshold = 0.5   # adjust as needed

# Containers for per‐group graphs
group_graphs = {}

# Iterate over each leiden_t_bin_merged_nicer group column
for group in expr_mean.columns[:-2]:   # skip the last two columns (UniProtID, gene_symbol)
    # 1) Determine which proteins are “expressed” in this group
    expressed = set(
        expr_mean.loc[expr_mean[group] > expression_threshold, "UniProtID"]
    )
    print(f"{group}: {len(expressed)} proteins above threshold {expression_threshold}")
    
    # 2) Filter interactions to those among expressed proteins
    df_sub = df[
        df["source"].isin(expressed) &
        df["target"].isin(expressed)
    ].copy()
    print(f"  → {df_sub.shape[0]} edges remain in subnetwork")
    
    # 3) Build the directed subgraph
    edge_tuples = df_sub[["source","target","type"]].itertuples(index=False, name=None)
    g_sub = ig.Graph.TupleList(
        edge_tuples,
        directed=True,
        edge_attrs=["type"]
    )
    
    # 4) Annotate nodes with gene symbols
    g_sub.vs["gene_symbol"] = [symbol_map.get(v["name"], "") for v in g_sub.vs]
    
    # 5) Store in dict
    group_graphs[group] = g_sub

# Example: inspect one group's graph
grp = list(group_graphs)[0]
print(f"\nSubnetwork for {grp}: nodes={group_graphs[grp].vcount()}, edges={group_graphs[grp].ecount()}")
for v in group_graphs[grp].vs[:5]:
    print(f"  {v['name']} ({v['gene_symbol']})")


In [ ]:
g

## 6.2 Topological metrices (node and edge counts, centrlity) per state specific network

### Node and Edges Count

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- 1. Compute metrics per group subgraph ------------------
metrics = []
for grp, sub in group_graphs.items():
    n     = sub.vcount()
    m     = sub.ecount()
    dens  = sub.density()
    avgd  = np.mean(sub.degree()) if n>0 else 0
    metrics.append((grp, n, m, dens, avgd))

metrics = pd.DataFrame(metrics, columns=["group","nodes","edges","density","avg_deg"])
metrics = metrics.sort_values("group")

# Show the table
print(metrics)

# --- 2. Plot each metric in its own figure ---------------
for metric in ["nodes","edges","density","avg_deg"]:
    plt.figure()
    plt.bar(metrics["group"], metrics[metric])
    plt.xticks(rotation=90)
    plt.ylabel(metric.replace("_"," ").title())
    plt.title(f"{metric.replace('_',' ').title()} per Group")
    plt.tight_layout()
    plt.show()



In [ ]:
g

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

# --- 1. Compute metrics per group subgraph ------------------
metrics = []
for grp, sub in group_graphs.items():
    n     = sub.vcount()
    m     = sub.ecount()
    dens  = sub.density()
    avgd  = np.mean(sub.degree()) if n > 0 else 0
    metrics.append((grp, n, m, dens, avgd))

metrics = pd.DataFrame(metrics, columns=["group", "nodes", "edges", "density", "avg_deg"])
metrics = metrics.sort_values("group")

# --- 2. Define trajectories ------------------
trajectories = {
    "Traj_1": ["0_t_0.0000-0.5000", "1_t_0.5000-1.5000", "1_t_1.5000-2.5000", "1_t_2.5000-3.0000", "1_t_3.0000-3.5000"],
    "Traj_2": ["0_t_0.0000-0.5000", "3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000"],
    "Traj_3": ["3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000", "2_t_1.0000-2.5000", 
               "2_t_2.5000-3.0000", "2_t_3.0000-3.5000", "2_t_3.5000-4.0000"]
}

# --- 3. Create subpanels: one row per metric, one column per trajectory ---
metrics_to_plot = ["nodes", "edges", "density", "avg_deg"]
n_metrics = len(metrics_to_plot)
n_trajectories = len(trajectories)

fig, axes = plt.subplots(nrows=n_metrics, ncols=n_trajectories, figsize=(5 * n_trajectories, 4 * n_metrics), sharex=False)

for row_idx, metric in enumerate(metrics_to_plot):
    for col_idx, (traj_name, traj_groups) in enumerate(trajectories.items()):
        ax = axes[row_idx, col_idx]

        traj_df = metrics[metrics["group"].isin(traj_groups)].copy()
        traj_df["group"] = pd.Categorical(traj_df["group"], categories=traj_groups, ordered=True)
        traj_df = traj_df.sort_values("group")

        ax.plot(traj_df["group"], traj_df[metric], marker='o', linewidth=2)
        ax.set_title(f"{metric.replace('_', ' ').title()} – {traj_name}")
        ax.set_ylabel(metric.replace("_", " ").title())
        ax.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()


In [ ]:
g

### Degree distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress

# --- Define trajectories ---
trajectories = {
    "Traj_1": ["0_t_0.0000-0.5000", "1_t_0.5000-1.5000", "1_t_1.5000-2.5000", "1_t_2.5000-3.0000", "1_t_3.0000-3.5000"],
    "Traj_2": ["0_t_0.0000-0.5000", "3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000"],
    "Traj_3": ["3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000", "2_t_1.0000-2.5000", 
               "2_t_2.5000-3.0000", "2_t_3.0000-3.5000", "2_t_3.5000-4.0000"]
}

# --- Plot normalized degree distributions in log-log scale ---
for traj_name, traj_groups in trajectories.items():
    plt.figure(figsize=(8, 6))

    for group in traj_groups:
        if group not in group_graphs:
            continue

        g_sub = group_graphs[group]
        degrees = g_sub.degree()

        # Bin counts and normalize
        counts = np.bincount(degrees)
        if counts.sum() == 0:
            continue
        probs = counts / counts.sum()
        x = np.arange(len(probs))

        # Filter out zero values to avoid log(0)
        nonzero = probs > 0
        x_log = x[nonzero]
        probs_log = probs[nonzero]

        plt.plot(x_log, probs_log, marker='o', linestyle='-', label=group)

        # Fit tail to power law: P(k) ~ k^-alpha (e.g., for k >= 3)
        tail_mask = (x_log >= 3)
        if np.sum(tail_mask) > 2:
            slope, intercept, _, _, _ = linregress(np.log10(x_log[tail_mask]), np.log10(probs_log[tail_mask]))
            alpha = -slope
            plt.text(
                x_log[-1], probs_log[-1], f"$\\alpha$={alpha:.2f}", fontsize=8,
                ha="right", va="bottom"
            )

    plt.title(f"Normalized Degree Distribution (Log–Log) – {traj_name}")
    plt.xlabel("Degree (log scale)")
    plt.ylabel("Proportion of Nodes (log scale)")
    plt.xscale("log")
    plt.yscale("log")
    plt.grid(True, which="both", linestyle='--', alpha=0.5)
    plt.legend(fontsize="small", loc="upper right")
    plt.tight_layout()
    plt.show()


In [ ]:
g

### Betweeness centrality

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import linregress
from collections import defaultdict

In [ ]:
# ---------- Settings ----------
OUTDIR = "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/network"
N_SHOW = 10      # show top-10 per image
N_COLOR = 20     # assign fixed colors to overall top-20
DPI = 600
SHOW = True
YLIM = (0, 5100)

# High-quality, minimal style
mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": DPI,
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "lines.linewidth": 2.0,
    "lines.markersize": 4.5,
})
os.makedirs(OUTDIR, exist_ok=True)

# ---------- Your trajectories ----------
trajectories = {
    "Traj_1": ["0_t_0.0000-0.5000", "1_t_0.5000-1.5000", "1_t_1.5000-2.5000", "1_t_2.5000-3.0000", "1_t_3.0000-3.5000"],
    "Traj_2": ["0_t_0.0000-0.5000", "3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000"],
    "Traj_3": ["0_t_0.0000-0.5000", "3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000",
               "2_t_1.0000-2.5000", "2_t_2.5000-3.0000", "2_t_3.0000-3.5000", "2_t_3.5000-4.0000"]
}

# ---------- Step 1: Collect betweenness ----------
betw_data = []
for group, graph in group_graphs.items():
    btwn = graph.betweenness(directed=True)
    for v, score in zip(graph.vs, btwn):
        betw_data.append({"group": group, "gene_symbol": v["gene_symbol"], "betweenness": score})
betw_df = pd.DataFrame(betw_data)

# ---------- Helpers ----------
def slopes_for_trajectory(bdf, traj_groups):
    df = bdf[bdf["group"].isin(traj_groups)].copy()
    df["group"] = pd.Categorical(df["group"], categories=traj_groups, ordered=True)
    pivot = df.pivot_table(index="gene_symbol", columns="group", values="betweenness", fill_value=0)
    tp = np.arange(len(traj_groups))
    slope_records = []
    for gene, row in pivot.iterrows():
        y = row[traj_groups].values.astype(float)
        slope, intercept, r, pval, stderr = linregress(tp, y)
        slope_records.append((gene, slope, pval))
    slopes = pd.DataFrame(slope_records, columns=["gene_symbol", "slope", "pval"]).set_index("gene_symbol")
    return pivot, slopes

# ---------- Step 2: Compute slopes per trajectory ----------
traj_pivots, traj_slopes = {}, {}
for tname, tgroups in trajectories.items():
    pivot, slopes = slopes_for_trajectory(betw_df, tgroups)
    traj_pivots[tname] = pivot
    traj_slopes[tname] = slopes

# ---------- Step 3: Overall top genes by max |slope| across all trajectories ----------
all_slopes = defaultdict(list)
for tname, sdf in traj_slopes.items():
    for gene, s in sdf["slope"].items():
        all_slopes[gene].append(abs(s))
overall = pd.Series({gene: max(vals) for gene, vals in all_slopes.items()}).sort_values(ascending=False)

# Fixed color list = overall top-20
overall_top20 = list(overall.head(N_COLOR).index)

# ---------- Step 4: Color mapping ----------
# 20 distinct colors for the overall top-20 (tab20 has exactly 20 categorical colors)
cmap = plt.get_cmap("tab20")
gene_colors = {gene: cmap(i % 20) for i, gene in enumerate(overall_top20)}
FALLBACK = (0.6, 0.6, 0.6)  # gray for genes not in the overall top-20

# (Optional) Save the mapping for reproducibility
pd.Series(gene_colors).apply(lambda c: tuple(np.round(np.array(c[:3]), 3))).to_csv(
    os.path.join(OUTDIR, "color_map_overall_top20.csv"), header=False
)

# ---------- Step 5: Plot (show only top-10 per trajectory), save, and show ----------
saved_paths = []
for tname, tgroups in trajectories.items():
    pivot = traj_pivots[tname]

    # Slopes for this trajectory (Series indexed by gene_symbol)
    local_slopes = traj_slopes[tname]["slope"]

    # Pick top-10 by |slope| within this trajectory (only genes present in the pivot)
    local_abs = local_slopes.abs().sort_values(ascending=False)
    genes_to_plot = [gene for gene in local_abs.index if gene in pivot.index][:N_SHOW]

    # Order legend/lines by |slope| within this trajectory
    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    for gene in genes_to_plot:
        y = pivot.loc[gene, tgroups].values.astype(float)
        slope_val = float(local_slopes.loc[gene]) if gene in local_slopes.index else np.nan

        line, = ax.plot(
            tgroups, y,
            marker="o", linewidth=2.0, markersize=4.5,
            label=f"{gene} (slope={slope_val:+.3g})"
        )
        line.set_color(gene_colors.get(gene, FALLBACK))  # fixed color for overall top-20; gray otherwise

    ax.set_title(f"Betweenness Trends — {tname} (top {N_SHOW} in this trajectory)")
    ax.set_xlabel("Group (pseudotime)")
    ax.set_ylabel("Betweenness centrality")

    # Fixed y-axis across all plots
    ax.set_ylim(*YLIM)
    ax.set_yticks(np.arange(YLIM[0], YLIM[1] + 1, 1000))
    ax.tick_params(axis="x", rotation=40)

    # Cosmetics
    ax.grid(True, which="major", axis="y", linewidth=0.4, alpha=0.3)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    # Legend shows gene + slope
    ax.legend(title="Genes (with slope)", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    fig.tight_layout()

    fname = f"network_betweenness_trends_{tname}_top{N_SHOW}.png"
    outpath = os.path.join(OUTDIR, fname)
    fig.savefig(outpath, bbox_inches="tight", transparent=True)
    saved_paths.append(outpath)

    if SHOW:
        plt.show()
    plt.close(fig)


print("Saved figures:")
for p in saved_paths:
    print(" -", p)

print("\nOverall top-20 (fixed colors assigned):")
for i, gene in enumerate(overall_top20, 1):
    print(f"{i:>2}. {gene}")


In [ ]:
g

# 7. Calculate clusters of the network 

In [ ]:
### 7. 
#take the base network or graph g, and calculate modules with MCL clustering. 
#from this make a list of node names (uniprot, gene_symbol), and the annotated MCl clister
#if symbol is not present  gte it from expr_mean
#e.g symbol_map = adata_full.var.set_index("ensembl_gene_id")["gene_symbol"].to_dict()
#expr_mean["UniProtID"]   = expr_mean.index.map(ensg2up)
#expr_mean["gene_symbol"] = expr_mean.index.map(symbol_map)
#from expr_mean get also the log2chnages and group names (columns names)
#format everything that it is compatible with gsea fgsea for python, like a typical gene set format for the gsea analysis
#than iterate the gsea through the different 

#from clusters, get a list of gene name sand cluster zugehörigkeit, and than use it as gene set for gsea, 
#and use the expression values from expre_mean or adata, to calculate enirhcment of the cluster at differnt groups as column name sin expr_mean indicate. 
#get the enrichment results and store them on analysis folder /home/job37y/Projects_shared/PANC_cancer/analysis/network_module_gsea full path or relative pat from 
# /home/job37y/Projects_shared/PANC_cancer/code/scripts_beta

In [ ]:
g

In [ ]:
edge_list = []
for e in g.es:
    src, tgt = g.vs[e.tuple[0]]["name"], g.vs[e.tuple[1]]["name"]
    edge_list.append({
        "source": src,
        "target": tgt,
        "type":   e["type"]
    })
edges = pd.DataFrame(edge_list)
print(edges.head())

## 7.1 Test different inlfation values MCL cluster methods and compare modularity and number of modueles

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from markov_clustering import run_mcl, get_clusters

########################################
# 0.  Restrict to the giant component
########################################
components =g.clusters(mode="weak")
g0 = components.giant()   # keep only the largest connected subgraph
print("Giant comp: nodes", g0.vcount(), "edges", g0.ecount())
# map back to your original g.vs indices if needed later…


# --- Compare full graph vs giant component ---
print(f"Original graph: {g.vcount()} nodes, {g.ecount()} edges")
print(f"Giant component: {g0.vcount()} nodes, {g0.ecount()} edges")

# Optionally also the % retained
node_pct = 100 * g0.vcount() / g.vcount()
edge_pct = 100 * g0.ecount() / g.ecount()
print(f"Retained in giant component: {node_pct:.1f}% nodes, {edge_pct:.1f}% edges")



########################################
# 1.  A helper to run MCL + eval
########################################
def evaluate_mcl(graph, inflation):
    # get adjacency as array
    mx = np.array(graph.get_adjacency(attribute=None).data, dtype=float)
    pr = run_mcl(mx, inflation=inflation)
    clusters = get_clusters(pr)
    # build a flat module_id list
    module_id = np.zeros(graph.vcount(), dtype=int)
    for cid, members in enumerate(clusters):
        for vid in members:
            module_id[vid] = cid
    # compute modularity
    mod = graph.modularity(module_id)
    return len(clusters), mod, clusters

########################################
# 2.  Sweep inflation values
########################################
inflations = np.linspace(1.1, 3.0, 30)
results = []
for infl in inflations:
    ncl, mod_score, _ = evaluate_mcl(g0, inflation=infl)
    results.append((infl, ncl, mod_score))
results = np.array(results, dtype=float)
    
# plot
fig, ax1 = plt.subplots(figsize=(6,4))
ax1.plot(results[:,0], results[:,1],  marker='o', label="n_modules")
ax1.set_xlabel("Inflation")
ax1.set_ylabel("Number of modules", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(results[:,0], results[:,2],  marker='s', color="tab:red", label="modularity")
ax2.set_ylabel("Modularity", color="tab:red")
ax1.set_title("MCL clustering quality vs inflation")
fig.tight_layout()
plt.show()

########################################
# 3.  Choose an inflation & re‐run
########################################
best_infl = 1.5   # pick where modularity peaks or modules drop to your taste
ncl, mod_score, clusters = evaluate_mcl(g0, best_infl)
print(f"At inflation={best_infl:.2f}: modules={int(ncl)}, modularity={mod_score:.3f}")

# attach new modules back to g0 (or original g)
module_id = np.zeros(g0.vcount(), dtype=int)
for cid,members in enumerate(clusters):
    for vid in members:
        module_id[vid] = cid
g0.vs["module"] = module_id

########################################
# 4.  Per-module density & inter-module edges
########################################
densities = []
for cid, members in enumerate(clusters):
    sub = g0.induced_subgraph(members)
    densities.append(sub.density())
    
plt.figure(figsize=(5,3))
plt.hist(densities, bins=20, edgecolor="black")
plt.xlabel("Within-module density")
plt.ylabel("Module count")
plt.title("Distribution of intra-module densities")
plt.tight_layout()
plt.show()


## 7.2 Compare different cluster methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from markov_clustering import run_mcl, get_clusters
import igraph as ig
import leidenalg

# ─── 0.  Largest weakly‐connected component ─────────────────────────────────
comps = g.clusters(mode="weak")
g_weak = comps.giant()
print(f"Weakly-connected giant: {g_weak.vcount()} nodes, {g_weak.ecount()} edges")

# Prepare an undirected copy for algorithms that require it
g_undir = g_weak.as_undirected(combine_edges=None)

# ─── Helpers ──────────────────────────────────────────────────────────────────
def mcl_part(graph, inflation):
    mx = np.array(graph.get_adjacency(attribute=None).data, dtype=float)
    res = run_mcl(mx, inflation=inflation)
    cl = get_clusters(res)
    mem = np.zeros(graph.vcount(), dtype=int)
    for cid, members in enumerate(cl):
        for vid in members:
            mem[vid] = cid
    return cl, mem

def louvain_part(graph):
    part = graph.community_multilevel()
    return part, np.array(part.membership)

def leiden_part(graph, resolution=1.0):
    part = leidenalg.find_partition(
        graph,
        leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=resolution
    )
    return part, np.array(part.membership)

def infomap_part(graph):
    part = graph.community_infomap(edge_weights=None, trials=10)
    return part, np.array(part.membership)

def metrics(graph, clusters, membership):
    n_mod = len(clusters)
    mod_sc = graph.modularity(membership)
    dens = [graph.induced_subgraph(c).density() for c in clusters]
    return n_mod, mod_sc, np.mean(dens)

# ─── 1.  Run clustering ────────────────────────────────────────────────────────
results = []

# 1a) Infomap (directed)
inf_cl, inf_mem = infomap_part(g_weak)
results.append(("Infomap",) + metrics(g_weak, inf_cl, inf_mem))

# 1b) MCL on UNDIRECTED weak component
mcl_cl1, mcl_mem1 = mcl_part(g_undir, inflation=1.2)
results.append(("MCL-1.2",) + metrics(g_undir, mcl_cl1, mcl_mem1))

mcl_cl2, mcl_mem2 = mcl_part(g_undir, inflation=1.6)
results.append(("MCL-1.6",) + metrics(g_undir, mcl_cl2, mcl_mem2))

# 1c) Louvain (undirected)
louv_cl, louv_mem = louvain_part(g_undir)
results.append(("Louvain",) + metrics(g_undir, louv_cl, louv_mem))

# 1d) Leiden (undirected)
lei_cl, lei_mem = leiden_part(g_undir, resolution=1.0)
results.append(("Leiden",) + metrics(g_undir, lei_cl, lei_mem))

# ─── 2.  Summarize results ────────────────────────────────────────────────────
names, nmods, mods, dens = zip(*results)
x = np.arange(len(names))

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].bar(x, nmods);   axs[0].set_xticks(x);   axs[0].set_xticklabels(names, rotation=45);   axs[0].set_title("# Modules")
axs[1].bar(x, mods, color="tab:red"); axs[1].set_xticks(x); axs[1].set_xticklabels(names, rotation=45); axs[1].set_title("Modularity")
axs[2].bar(x, dens, color="tab:green"); axs[2].set_xticks(x); axs[2].set_xticklabels(names, rotation=45); axs[2].set_title("Mean Density")
plt.tight_layout()
plt.show()

# ─── 3.  Degree distributions ─────────────────────────────────────────────────
indeg = g_weak.indegree()
outdeg = g_weak.outdegree()

plt.figure(figsize=(6, 4))
plt.hist(indeg, bins=50, alpha=0.7, label="In-degree")
plt.hist(outdeg, bins=50, alpha=0.7, label="Out-degree")
plt.yscale("log")
plt.xlabel("Degree")
plt.ylabel("Count (log scale)")
plt.title("Directed Degree Distributions (Weak Component)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from markov_clustering import run_mcl, get_clusters
import igraph as ig
import leidenalg
from collections import Counter

# ─── Prepare the giant component & undirected copy ────────────────────────────
comps   = g.components(mode="weak")
g0      = comps.giant()
g_undir = g0.as_undirected(combine_edges=None)

# ─── Clustering functions ─────────────────────────────────────────────────────
def mcl_clusters(graph, inflation):
    mx      = np.array(graph.get_adjacency(attribute=None).data, dtype=float)
    res     = run_mcl(mx, inflation=inflation)
    cls     = get_clusters(res)
    return cls

def infomap_clusters(graph):
    part    = graph.community_infomap(edge_weights=None, trials=10)
    # igraph returns a VertexClustering object; convert to list of lists
    return [c for c in part]

def louvain_clusters(graph):
    part    = graph.community_multilevel()
    return [c for c in part]

def leiden_clusters(graph, resolution=1.0):
    part    = leidenalg.find_partition(
                  graph,
                  leidenalg.RBConfigurationVertexPartition,
                  resolution_parameter=resolution
              )
    return [ [v.index for v,m in zip(graph.vs, part.membership) if m==cid]
             for cid in set(part.membership) ]

# ─── Compute clusterings ──────────────────────────────────────────────────────
methods = [
    ("MCL 1.2",  lambda: mcl_clusters(g_undir, 1.2)),
    ("MCL 1.6",  lambda: mcl_clusters(g_undir, 1.6)),
    ("MCL 2.0",  lambda: mcl_clusters(g_undir, 2.0)),
    ("Infomap",  lambda: infomap_clusters(g0)),
    ("Louvain",  lambda: louvain_clusters(g_undir)),
    ("Leiden",   lambda: leiden_clusters(g_undir, resolution=1.0)),
]

# ─── Plot size distributions ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15,10))
axes = axes.flatten()

for ax, (name, fn) in zip(axes, methods):
    clusters = fn()
    sizes    = [len(c) for c in clusters]
    median   = np.median(sizes)
    p10, p90 = np.percentile(sizes, [10, 90])
    
    ax.hist(sizes, bins=30, edgecolor="black")
    ax.set_yscale("log")
    ax.set_title(f"{name}\nmodules={len(sizes)}")
    ax.axvline(median, color="red", linestyle="--", label=f"median={median:.0f}")
    ax.axvline(p10, color="orange", linestyle=":", label=f"10th={p10:.0f}")
    ax.axvline(p90, color="orange", linestyle=":", label=f"90th={p90:.0f}")
    ax.set_xlabel("Genes per module")
    ax.set_ylabel("Module count (log)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 7.3 Choose a cluster method and annotate g with modules and export

In [ ]:
# ===== Choose clustering method for module annotation =====
# Options: "infomap", "louvain", "leiden", "mcl"
CLUSTER_METHOD = "leiden"

# Optional params by method
LEIDEN_RESOLUTION = 1.0
MCL_INFLATION     = 1.6     # typical 1.2–2.5

# ===== Imports =====
import numpy as np
import pandas as pd
import igraph as ig
import leidenalg
from markov_clustering import run_mcl, get_clusters

# ===== 0) Giant weakly connected component (keeps g safe) =====
comps = g.components(mode="weak")
g0 = comps.giant()                           # work graph
print(f"weak GCC: {g0.vcount()} nodes, {g0.ecount()} edges")

# Ensure a good 'name' and gene_symbol columns
if "name" not in g0.vs.attribute_names():
    g0.vs["name"] = [f"v{i}" for i in range(g0.vcount())]
if "gene_symbol" not in g0.vs.attribute_names():
    g0.vs["gene_symbol"] = g0.vs["name"]

# Undirected copy for methods that need it
g_undir = g0.as_undirected(combine_edges=None)

# ===== 1) Clustering runners (return membership: list[int] length = vcount) =====
def run_infomap_directed(G: ig.Graph):
    part = G.community_infomap(edge_weights=None, trials=10)
    return part.membership

def run_louvain_undirected(G: ig.Graph):
    part = G.community_multilevel()
    return list(part.membership)

def run_leiden_undirected(G: ig.Graph, resolution=1.0):
    part = leidenalg.find_partition(
        G, leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=resolution
    )
    return list(part.membership)

def run_mcl_undirected(G: ig.Graph, inflation=1.6):
    # adjacency -> MCL -> clusters -> membership
    mx = np.array(G.get_adjacency(attribute=None).data, dtype=float)
    res = run_mcl(mx, inflation=inflation)
    clusters = get_clusters(res)  # list of lists of vertex indices
    membership = np.zeros(G.vcount(), dtype=int)
    for cid, members in enumerate(clusters):
        for vid in members:
            membership[vid] = cid
    return membership.tolist()

# ===== 2) Pick method & compute membership =====
method = CLUSTER_METHOD.lower()
if method == "infomap":
    membership = run_infomap_directed(g0)            # directed
elif method == "louvain":
    membership = run_louvain_undirected(g_undir)     # undirected
elif method == "leiden":
    membership = run_leiden_undirected(g_undir, resolution=LEIDEN_RESOLUTION)
elif method == "mcl":
    membership = run_mcl_undirected(g_undir, inflation=MCL_INFLATION)
else:
    raise ValueError("CLUSTER_METHOD must be one of: infomap, louvain, leiden, mcl")

# Annotate modules
g0.vs["module"] = membership
n_modules = len(set(membership))
print(f"[{CLUSTER_METHOD}] modules = {n_modules}")

# ===== 3) Build vertex & edge tables; print head =====
vertices = pd.DataFrame({
    "UniProtID":   g0.vs["name"],
    "gene_symbol": g0.vs["gene_symbol"],
    "module":      g0.vs["module"]
})
print("\nVertices (head):")
print(vertices.head(10).to_string(index=False))

edge_rows = []
etype_present = "type" in g0.es.attribute_names()
for e in g0.es:
    s, t = e.tuple
    edge_rows.append({
        "source": g0.vs[s]["name"],
        "target": g0.vs[t]["name"],
        "type":   e["type"] if etype_present else "interacts_with"
    })
edges = pd.DataFrame(edge_rows)
print("\nEdges (head):")
print(edges.head(10).to_string(index=False))

# ===== 4) Hand back the annotated giant as your working graph =====
g = g0


In [ ]:
import pandas as pd

# --- 1. List available attributes -----------------------
print("Vertex attributes:", g.vs.attribute_names())
print("Edge attributes:  ", g.es.attribute_names())

# --- 2. Pull out a DataFrame of vertices ---------------
vertices = pd.DataFrame({
    "UniProtID":    g.vs["name"],
    "gene_symbol":  g.vs["gene_symbol"],
    "module":  g.vs["module"],
})
print(vertices.head())

# --- 3. Pull out a DataFrame of edges ------------------
# Note: igraph edges don’t automatically carry source/target names in a column,
#       so we reconstruct them via the endpoints of each edge.
edge_list = []
for e in g.es:
    src, tgt = g.vs[e.tuple[0]]["name"], g.vs[e.tuple[1]]["name"]
    edge_list.append({
        "source": src,
        "target": tgt,
        "type":   e["type"]
    })
edges = pd.DataFrame(edge_list)
print(edges.head())


In [ ]:
# ─── Export as GraphML (keeps igraph attributes intact) ───────────────────────
outdir = "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea"
os.makedirs(outdir, exist_ok=True)
graphml_path = os.path.join(outdir, "network_modules.graphml")
g.write_graphml(graphml_path)
print("Wrote igraph GraphML to", graphml_path)

## 7.4 Viusalize network and modules: top modules

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
from collections import Counter
import numpy as np

# --- Build full NetworkX graph from igraph g ---
nxg = nx.DiGraph()
for v in g.vs:
    up = v["name"]
    nxg.add_node(up,
                 module=v["module"],
                 gene_symbol=symbol_map.get(up, up))  # gene name fallback
for e in g.es:
    src = g.vs[e.tuple[0]]["name"]
    tgt = g.vs[e.tuple[1]]["name"]
    nxg.add_edge(src, tgt, type=e["type"])

# --- Identify top-4 modules by size ---
mod_counts = Counter(nx.get_node_attributes(nxg, "module").values())
top4 = [m for m, _ in mod_counts.most_common(8)]

# --- Assign consistent module colors ---
palette = plt.cm.tab10(np.linspace(0, 1, len(top4)))
mod2color = {mod: rgb2hex(palette[i]) for i, mod in enumerate(top4)}

# --- Prepare subplots ---
fig, axs = plt.subplots(2, 2, figsize=(14, 12))
axs = axs.flatten()

for ax, mod in zip(axs, top4):
    # Subgraph for this module
    members = [n for n, d in nxg.nodes(data=True) if d["module"] == mod]
    subg = nxg.subgraph(members)
    
    # Use better layout to reduce overlap
    pos = nx.kamada_kawai_layout(subg)  # better than spring for small graphs

    # Assign module color to all nodes in this subgraph
    node_colors = [mod2color[mod]] * len(subg.nodes)

    # Show labels only for high-degree nodes
    degrees = dict(subg.degree())
    threshold = np.percentile(list(degrees.values()), 5)
    top_deg_nodes = [n for n, d in degrees.items() if d >= threshold]
    labels = {n: subg.nodes[n]["gene_symbol"] for n in top_deg_nodes}

    # Split edges by type
    act_edges = [(u, v) for u, v, d in subg.edges(data=True) if d["type"] == "activation"]
    inh_edges = [(u, v) for u, v, d in subg.edges(data=True) if d["type"] == "inhibition"]

    # Draw nodes
    nx.draw_networkx_nodes(subg, pos,
                           node_color=node_colors,
                           node_size=300,
                           ax=ax)

    # Draw activation edges
    nx.draw_networkx_edges(subg, pos,
                           edgelist=act_edges,
                           edge_color="green",
                           arrowstyle='->',
                           arrowsize=8,
                           alpha=0.8,
                           ax=ax)

    # Draw inhibition edges
    nx.draw_networkx_edges(subg, pos,
                           edgelist=inh_edges,
                           edge_color="red",
                           arrowstyle='-|>',
                           arrowsize=8,
                           alpha=0.8,
                           ax=ax)

    # Draw readable labels (high-degree only)
    nx.draw_networkx_labels(subg, pos,
                            labels=labels,
                            font_size=7,
                            font_color="black",
                            bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.2"),
                            ax=ax)

    ax.set_title(f"Module {mod} (n={mod_counts[mod]})")
    ax.axis("off")

# --- Add global legend ---
handles = [
    plt.Line2D([0], [0], color="green", lw=2, label="activation"),
    plt.Line2D([0], [0], color="red", lw=2, label="inhibition"),
]
axs[1].legend(handles=handles, loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
def get_module_for_gene(G, gene_symbol):
    """Return module ID(s) for a given gene_symbol in graph G."""
    if "gene_symbol" not in G.vs.attribute_names():
        raise ValueError("Graph does not have a 'gene_symbol' attribute.")
    matches = [v["module"] for v in G.vs if v["gene_symbol"] == gene_symbol]
    return set(matches) if matches else None

# Examples:
print("SMAD3 →", get_module_for_gene(g, "SMAD3"))
print("WEE1  →", get_module_for_gene(g, "WEE1"))
print("CDK1  →", get_module_for_gene(g, "CDK1"))
print("CDKN1A  →", get_module_for_gene(g, "CDKN1A"))

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
import numpy as np

# ─── Convert igraph → NetworkX with annotations ──────────────────────────────
nxg = nx.DiGraph()

for v in g.vs:
    ensg = v["name"]
    nxg.add_node(ensg,
                 module=v["module"],
                 gene_symbol=v["gene_symbol"])  # taken directly from g.vs

for e in g.es:
    src = g.vs[e.source]["name"]
    tgt = g.vs[e.target]["name"]
    nxg.add_edge(src, tgt, type=e["type"])

# ─── Select module and subgraph ───────────────────────────────────────────────
mod_id = 9
members = [n for n, d in nxg.nodes(data=True) if d["module"] == mod_id]
subg = nxg.subgraph(members).copy()

# ─── Layout ───────────────────────────────────────────────────────────────────
pos = nx.spring_layout(subg, seed=42, k=0.4)

# ─── Node coloring (uniform by module) ────────────────────────────────────────
node_color = rgb2hex(plt.cm.tab10(mod_id % 1))
node_colors = [node_color] * len(subg.nodes)

# ─── Labels: show gene_symbol ─────────────────────────────────────────────────
labels = {n: d["gene_symbol"] for n, d in subg.nodes(data=True)}

# ─── Edge types (activation/inhibition) ───────────────────────────────────────
act_edges = [(u, v) for u, v, d in subg.edges(data=True) if d["type"] == "activation"]
inh_edges = [(u, v) for u, v, d in subg.edges(data=True) if d["type"] == "inhibition"]

# ─── Plot ─────────────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 10))

# Draw nodes
nx.draw_networkx_nodes(subg, pos,
                       node_color=node_colors,
                       node_size=300,
                       alpha=0.9)

# Draw activation edges (green)
nx.draw_networkx_edges(subg, pos,
                       edgelist=act_edges,
                       edge_color="green",
                       arrowstyle='->',
                       arrowsize=8,
                       alpha=0.7)

# Draw inhibition edges (red)
nx.draw_networkx_edges(subg, pos,
                       edgelist=inh_edges,
                       edge_color="red",
                       arrowstyle='-|>',
                       arrowsize=8,
                       alpha=0.7)

# Draw gene symbol labels
nx.draw_networkx_labels(subg, pos,
                        labels=labels,
                        font_size=7,
                        font_color="black",
                        bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.2"))

plt.title(f"Module {mod_id} (n={len(subg.nodes)})")
plt.axis("off")
plt.tight_layout()
plt.show()


## 7.5 Save graph object g

In [ ]:
import pandas as pd
import igraph as ig

# ----------------------------
# 1. Inspect attributes
# ----------------------------
print("📌 Vertex attributes:", g.vs.attributes())
print("📌 Edge attributes:", g.es.attributes())
print(f"🔢 Vertices: {g.vcount()}, Edges: {g.ecount()}")

# ----------------------------
# 2. Vertex attributes DataFrame
# ----------------------------
vertex_df = pd.DataFrame({attr: g.vs[attr] for attr in g.vs.attributes()})
print("\n🧬 Vertex table (all attributes):")
print(vertex_df.head(10))   # preview first 10

# ----------------------------
# 3. Edge attributes DataFrame
# ----------------------------
edge_attr_dict = {attr: g.es[attr] for attr in g.es.attributes()}
edge_df = pd.DataFrame(edge_attr_dict)

# Add endpoints
src = [e.tuple[0] for e in g.es]
tgt = [e.tuple[1] for e in g.es]
edge_df.insert(0, "source_idx", src)
edge_df.insert(1, "target_idx", tgt)

# Map to Ensembl IDs (or names)
if "name" in g.vs.attributes():
    idx2name = dict(enumerate(g.vs["name"]))
    edge_df["source_name"] = [idx2name[i] for i in src]
    edge_df["target_name"] = [idx2name[i] for i in tgt]

print("\n🕸️ Edge table (all attributes):")
print(edge_df.head(10))   # preview first 10

# ----------------------------
# 4. Save graph with attributes
# ----------------------------
g.write_graphml("/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_with_attributes.graphml")
print("\n💾 Saved graph as 'graph_with_attributes.graphml' (keeps all attributes).")

# Optional: also save CSVs for inspection
vertex_df.to_csv("/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_vertices.csv", index=False)
edge_df.to_csv("/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_edges.csv", index=False)
print("💾 Also saved 'graph_vertices.csv' and 'graph_edges.csv'")


# 8. Add node attributes: Expression values and add gene annotaiton (gene_symbol, enesembl_gene_id, uniprot_id)

## 8.1 Define a expr_mean table with Uniport_ID as leadign ID

In [ ]:
expr_mean

In [ ]:
import re
import numpy as np
import pandas as pd

# --- 0) Identify expression columns in expr_mean ---
EXPR_COLS = [c for c in expr_mean.columns if c not in {"UniProtID", "gene_symbol"}]

# --- 1) Make a clean, exploded table (handles many-to-many Ensembl↔UniProt) ---
def split_uniprot(s):
    if pd.isna(s):
        return []
    parts = re.split(r"[;,|\s]+", str(s).strip())
    return [p for p in parts if p]

# expr_mean has Ensembl IDs as index; make it a column
df = expr_mean.copy().rename_axis("ensembl_gene_id").reset_index()

# normalize UniProt and explode to one row per (Ensembl, UniProt)
df_exploded = (
    df.assign(UniProtID=lambda d: d["UniProtID"].apply(split_uniprot))
      .explode("UniProtID", ignore_index=True)
)
df_exploded = df_exploded[df_exploded["UniProtID"].notna() & (df_exploded["UniProtID"] != "")]
df_exploded = df_exploded.drop_duplicates(subset=["ensembl_gene_id", "UniProtID"])

# --- 2) Leading UniProt per Ensembl (pick the first) ---
first_uniprot = (
    df_exploded.sort_values(["ensembl_gene_id", "UniProtID"])
               .groupby("ensembl_gene_id", as_index=False)
               .first()[["ensembl_gene_id", "UniProtID"]]
)

# --- 3) Attach gene_symbol (from original df) ---
sym = df[["ensembl_gene_id", "gene_symbol"]].drop_duplicates("ensembl_gene_id")
expr_mean2 = first_uniprot.merge(sym, on="ensembl_gene_id", how="left")

# --- 4) Add cluster_id from the graph (use g.vs['module']) ---
# graph uses UniProt IDs in g.vs["name"]
if "module" in g.vs.attributes():
    g_df = pd.DataFrame({"UniProtID": g.vs["name"], "cluster_id": g.vs["module"]})
    expr_mean2 = expr_mean2.merge(g_df, on="UniProtID", how="left")
else:
    expr_mean2["cluster_id"] = np.nan  # if no module present

# --- 5) Add the expression columns to expr_mean2 ---
expr_mean2 = expr_mean2.merge(
    df[["ensembl_gene_id"] + EXPR_COLS],
    on="ensembl_gene_id",
    how="left"
)

# Reorder columns nicely
expr_mean2 = expr_mean2[
    ["ensembl_gene_id", "UniProtID", "gene_symbol", "cluster_id"] + EXPR_COLS
]

print("✅ Built expr_mean2 with leading UniProt per Ensembl.")
print(expr_mean2.head(10))

# --- 6) Add ensembl_gene_id to graph g ---
# Build UniProt → Ensembl (primary) from the exploded mapping
uid2ensg_list = (
    df_exploded.groupby("UniProtID")["ensembl_gene_id"]
               .apply(lambda s: list(dict.fromkeys(s)))
               .to_dict()
)
uid2ensg_primary = {u: (v[0] if len(v) else "") for u, v in uid2ensg_list.items()}

# Write attributes to graph (keep name = UniProt)
g.vs["uniprot_id"] = g.vs["name"]
g.vs["ensembl_gene_id_list"] = [uid2ensg_list.get(u, []) for u in g.vs["uniprot_id"]]
g.vs["ensembl_gene_id"]      = [uid2ensg_primary.get(u, "") for u in g.vs["uniprot_id"]]

print("✅ Graph updated with 'ensembl_gene_id' (primary) and 'ensembl_gene_id_list'.")

# --- 7) Coverage check: are all graph UniProt IDs in expr_mean/expr_mean2? ---
uids_graph = set(g.vs["uniprot_id"])
uids_expr2 = set(expr_mean2["UniProtID"])
missing_in_expr2 = sorted(uids_graph - uids_expr2)

print(f"📊 Coverage: {len(uids_graph) - len(missing_in_expr2)}/{len(uids_graph)} graph UniProt IDs found in expr_mean2.")
if missing_in_expr2:
    print("⚠️ Missing examples (up to 20):", missing_in_expr2[:20])


In [ ]:
# Get UniProt IDs from graph
uids_graph = set(g.vs["uniprot_id"])

# Filter expr_mean2 to keep only rows with UniProtID in graph
expr_mean2 = expr_mean2[expr_mean2["UniProtID"].isin(uids_graph)].copy()

print(f"✅ Filtered expr_mean2: {expr_mean2.shape[0]} rows remain (only UniProts in graph).")

# Preview
print(expr_mean2_filtered.head(10))

# Optional: save
expr_mean2_filtered.to_csv("/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea/expr_mean2.csv", index=False)
print("💾 Saved expr_mean2.csv")


In [ ]:
expr_mean2

## 8.1 Process Ggraph and Expression values

In [ ]:
#adata_sync = sc.read("/storage/users/data/PANC/H5AD_file/adata_full_sync.h5ad")
#adata_full = sc.read('/storage/users/data/PANC/H5AD_file/adata_filtered_no2D.h5ad')
adata_full = sc.read('/bigstorage/users/data/PANC/H5AD_file/adata_filtered_no2D.h5ad')

In [ ]:
adata_sync

## 8.2 make new adata objet based on graph nodes and graph modules/clusters

In [ ]:
import anndata as ad
import pandas as pd
import os

# 1) Read the CSV, telling pandas to NOT treat the first column as the index yet
expr_mean2 = pd.read_csv(
    "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea/expr_mean2.csv",
    header=0
)

# 2) Rename that unnamed first column to 'ensembl_gene_id'
#    (Pandas will call it 'Unnamed: 0' by default)
expr_mean2 = expr_mean2.rename(columns={"Unnamed: 0": "ensembl_gene_id"})

# 3) Drop any columns that end in '.1' (these are the duplicated UniProtID.1, gene_symbol.1)
expr_mean2 = expr_mean2.loc[:, ~expr_mean2.columns.str.endswith(".1")]

# 4) Set the new index
expr_mean2 = expr_mean2.set_index("ensembl_gene_id", drop=False)

# 5) (Optional) Reorder columns so index-like columns come first
cols = ["ensembl_gene_id", "UniProtID", "gene_symbol", "cluster_id"] + [
    c for c in expr_mean2.columns 
      if c not in ("ensembl_gene_id", "UniProtID", "gene_symbol", "cluster_id")
]
expr_mean2 = expr_mean2[cols]

# 6) Save back (or continue using 'expr_mean2` in memory)
expr_mean2.to_csv(
    "/storage/users/job37yv/Projects/PANC_cancer/analysis/"
    "network_module_gsea/expr_mean_filtered_with_cluster.cleaned.csv"
)

print(expr_mean2.head())
print("Columns:", expr_mean2.columns.tolist())


# 3) Build mapping dicts
ens_list = expr_mean2["ensembl_gene_id"].tolist()
ens2up   = dict(zip(expr_mean2["ensembl_gene_id"], expr_mean2["UniProtID"]))
ens2sym  = dict(zip(expr_mean2["ensembl_gene_id"], expr_mean2["gene_symbol"]))
ens2cl   = dict(zip(expr_mean2["ensembl_gene_id"], expr_mean2["cluster_id"]))

# 4) Subset adata_sync to only those Ensembl genes
keep = [gene for gene in adata_sync.var_names if gene in ens_list]
adata_sub = adata_sync[:, keep].copy()

# 5) Reorder to match expr_mean2 order
adata_sub = adata_sub[:, ens_list].copy()

# 6) Switch var_names to UniProtID and annotate var
uni_ids = [ens2up[ens] for ens in adata_sub.var_names]
adata_sub.var_names = ens_list
adata_sub.var["uniprot_id"] = uni_ids
adata_sub.var["ensembl_gene_id"] = ens_list
adata_sub.var["gene_symbol"]      = [ens2sym[ens] for ens in ens_list]
adata_sub.var["cluster_id"]       = [ens2cl[ens]  for ens in ens_list]

# 7) Save the new AnnData
out_path = (
    "/storage/users/job37yv/Projects/PANC_cancer/analysis"
    "/network_module_gsea/adata_sync_modules.h5ad"
)
os.makedirs(os.path.dirname(out_path), exist_ok=True)
adata_sub.write_h5ad(out_path)

print(f"New AnnData: {adata_sub.n_obs} cells × {adata_sub.n_vars} genes")
print("Var annotation columns:", list(adata_sub.var.columns))


In [ ]:
adata_sub

In [ ]:
adata_sub.write_h5ad("/bigstorage/users/data/PANC/H5AD_file/adata_sub.h5ad")

In [ ]:
adata_sub.var_names

## 8.3 Calculate Z-scores

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad

# 1) Load your AnnData subset
adata_sub = ad.read_h5ad(
    "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/"
    "network_module_gsea/adata_sync_modules.h5ad"
)

# 2) Define time bins based on group key
group_key = "leiden_t_bin_merged_nicer"
all_bins  = adata_sub.obs[group_key].cat.categories.tolist()

# 3) Pull dense expression matrix
X = adata_sub.X.toarray() if hasattr(adata_sub.X, "toarray") else adata_sub.X
genes = adata_sub.var_names.tolist()

# 4) Compute mean expression per gene × bin
mean_expr = pd.DataFrame(index=genes, columns=all_bins, dtype=float)
for b in all_bins:
    mask = adata_sub.obs[group_key] == b
    mean_expr[b] = X[mask, :].mean(axis=0)

# 5) Z-score per gene (row-wise normalization)
zscore = mean_expr.sub(mean_expr.mean(axis=1), axis=0)
zscore = zscore.div(mean_expr.std(axis=1), axis=0)

# 6) Save to CSV
z_path = (
    "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/"
    "network_module_gsea/zscore_by_group.csv"
)
zscore.to_csv(z_path)
print(f"✅ Saved z-score matrix ({zscore.shape[0]} genes × {zscore.shape[1]} bins) to:\n  {z_path}")
print(zscore.head())


## 8.4 Calculate log2fold changes

In [ ]:
adata_sync = sc.read("/bigstorage/users/data/PANC/H5AD_file/adata_full_sync.h5ad")

In [ ]:
adata_sync.var_names

In [ ]:
adata_sync

In [ ]:
adata_sync.var_names

In [ ]:
g

In [ ]:
import numpy as np
import pandas as pd

# ----------------------------
# 1. Extract gene IDs from both sources
# ----------------------------

# Genes in graph
genes_in_g = set(g.vs["ensembl_gene_id"])  # Ensembl gene IDs

# Genes in adata_sync
genes_in_adata = set(adata_sync.var_names)

# ----------------------------
# 2. Compare overlap
# ----------------------------
shared_genes = genes_in_g & genes_in_adata
n_total_g = len(genes_in_g)
n_total_adata = len(genes_in_adata)
n_shared = len(shared_genes)

print(f"🧠 Graph nodes:         {n_total_g}")
print(f"🧬 Genes in adata_sync: {n_total_adata}")
print(f"🔗 Shared Ensembl IDs:  {n_shared} ({n_shared / n_total_g:.1%} of graph genes)")

# ----------------------------
# 3. Subset adata_sync to those present in graph g
# ----------------------------
adata_sync_sub = adata_sync[:, adata_sync.var_names.isin(genes_in_g)].copy()
print(f"✅ Subsetted adata_sync_sub shape: {adata_sync_sub.shape} (cells × genes)")


In [ ]:
adata_sync_sub

In [ ]:
import scanpy as sc
import pandas as pd
from tqdm import tqdm

# Settings
group_key = "leiden_t_bin_merged_nicer"
baseline = "0_t_0.0000-0.5000"
groups_to_test = [
    "1_t_0.5000-1.5000", "1_t_1.5000-2.5000", "1_t_2.5000-3.0000", "1_t_3.0000-3.5000",
    "3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000",
    "2_t_1.0000-2.5000", "2_t_2.5000-3.0000", "2_t_3.0000-3.5000", "2_t_3.5000-4.0000"
]

# Optional: check group availability
available_groups = adata_sync_sub.obs[group_key].unique().tolist()
print("📦 Available groups in adata_sync_sub:", available_groups)

# Build mapping from Ensembl → gene symbol using expr_mean index
ensg_to_symbol = expr_mean["gene_symbol"].to_dict()

# Collect results
all_de_results = []

for group in tqdm(groups_to_test, desc="Running DE vs baseline"):
    # Subset data
    subset = adata_sync_sub[adata_sync_sub.obs[group_key].isin([baseline, group])].copy()
    subset.obs["compare"] = subset.obs[group_key]

    # Run differential expression
    sc.tl.rank_genes_groups(
        subset,
        groupby="compare",
        groups=[group],
        reference=baseline,
        method="t-test",
        use_raw=False
    )

    # Extract Ensembl IDs from result (they're stored as 'names')
    ensg_ids = subset.uns["rank_genes_groups"]["names"][group]

    # Build dataframe
    de = pd.DataFrame({
        "ensembl_gene_id": ensg_ids,
        "log2fc": subset.uns["rank_genes_groups"]["logfoldchanges"][group],
        "pval": subset.uns["rank_genes_groups"]["pvals"][group],
        "padj": subset.uns["rank_genes_groups"]["pvals_adj"][group],
    })
    de["gene_symbol"] = de["ensembl_gene_id"].map(ensg_to_symbol)
    de["bin"] = group
    de["baseline"] = baseline
    de["comparison"] = f"{group}_vs_{baseline}"

    all_de_results.append(de)

# Combine results
final_de_df = pd.concat(all_de_results, ignore_index=True)

# Save to CSV
out_path = "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea/de_genes_groupwise_vs_baseline.csv"
final_de_df.to_csv(out_path, index=False)

# Preview
print("✅ Differential expression results shape:", final_de_df.shape)
print(final_de_df.head(5))
print(f"📁 Saved to: {out_path}")


In [ ]:
print(final_de_df.head())

# 9. Viusalize network, subnetwork and modules

## 9.1 Visualize top hub network 

### color modules

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex

# 1) Pick top-50 hubs by degree and induce subgraph
deg   = g.degree()
top50 = sorted(range(g.vcount()), key=lambda i: deg[i], reverse=True)[:50]
subg  = g.induced_subgraph(top50)

# 2) Build NetworkX graph with module, type, and gene_symbol
nxg = nx.DiGraph()
for v in subg.vs:
    up   = v["name"]
    mod  = v["module"]
    sym  = symbol_map.get(up, up)   # map UniProt → gene_symbol
    nxg.add_node(up, module=mod, gene_symbol=sym)

for e in subg.es:
    src = subg.vs[e.tuple[0]]["name"]
    tgt = subg.vs[e.tuple[1]]["name"]
    t   = e["type"]
    nxg.add_edge(src, tgt, type=t)

# 3) Improved layout
pos = nx.kamada_kawai_layout(nxg)  # better spacing for dense subgraphs

# 4) Prepare node colors by module
mods       = nx.get_node_attributes(nxg, "module")
unique_mod = sorted(set(mods.values()))
palette    = plt.cm.tab20(np.linspace(0, 1, len(unique_mod)))
mod2col    = {m: rgb2hex(palette[i % len(palette)]) for i, m in enumerate(unique_mod)}
node_colors = [mod2col[mods[n]] for n in nxg.nodes()]

# 5) Split edges by type
act_edges = [(u, v) for u, v, d in nxg.edges(data=True) if d["type"] == "activation"]
inh_edges = [(u, v) for u, v, d in nxg.edges(data=True) if d["type"] == "inhibition"]

# 6) Draw
plt.figure(figsize=(10, 10))  # Larger plot
nx.draw_networkx_nodes(nxg, pos,
                       node_color=node_colors,
                       node_size=300,
                       alpha=0.95)

labels = nx.get_node_attributes(nxg, "gene_symbol")
nx.draw_networkx_labels(nxg, pos,
                        labels=labels,
                        font_size=7,
                        font_color="black",
                        verticalalignment="center")

# Draw edges with curvature for visibility
nx.draw_networkx_edges(nxg, pos,
                       edgelist=act_edges,
                       edge_color="green",
                       arrowstyle="->",
                       arrowsize=10,
                       alpha=0.5,
                       connectionstyle='arc3,rad=0.2')

nx.draw_networkx_edges(nxg, pos,
                       edgelist=inh_edges,
                       edge_color="red",
                       arrowstyle="-|>",
                       arrowsize=10,
                       alpha=0.5,
                       connectionstyle='arc3,rad=0.2')

plt.title("Top-50 hubs: modules (node colors) + activation (green) / inhibition (red)")
plt.axis("off")
plt.tight_layout()
plt.show()


### color log2fc

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, coolwarm
import pandas as pd

# ── SETTINGS ────────────────────────────────────────────────────────────────
baseline    = "0_t_0.0000-0.5000"   # DE reference
bins_to_plot = ["1_t_3.0000-3.5000", "3_t_1.0000-1.5000", "2_t_3.5000-4.0000"]  
sig_mask    = False   # if True, grey out padj>=0.05
number_of_hubs = 70

# final_de_df assumed to exist

# ── 0) Build UniProt → Ensembl & UniProt → gene_symbol from graph g ─────────
up2ensg   = {}
up2symbol = {}
for v in g.vs:
    up2ensg[v["name"]]   = v["ensembl_gene_id"] if "ensembl_gene_id" in v.attributes() else ""
    up2symbol[v["name"]] = v["gene_symbol"]     if "gene_symbol" in v.attributes() else v["name"]

# ── 1) Pivot DE table to wide log2FC [Ensembl × bin] ────────────────────────
de_sub = final_de_df[final_de_df["baseline"] == baseline]
log2fc_pivot = de_sub.pivot(index="ensembl_gene_id", columns="bin", values="log2fc")
padj_pivot   = de_sub.pivot(index="ensembl_gene_id", columns="bin", values="padj")

# sanity check bins
missing_bins = [b for b in bins_to_plot if b not in log2fc_pivot.columns]
if missing_bins:
    raise ValueError(f"Bins not found for baseline '{baseline}': {missing_bins}")

# ── 2) Pick top-50 hubs by degree in igraph ─────────────────────────────────
deg   = g.degree()
top50 = sorted(range(g.vcount()), key=lambda i: deg[i], reverse=True)[:number_of_hubs]
subg  = g.induced_subgraph(top50)

# ── 3) Convert to NetworkX keyed by UniProt ─────────────────────────────────
nxg = nx.DiGraph()
for v in subg.vs:
    up   = v["name"]
    mod  = v["module"]
    sym  = up2symbol.get(up, up)
    ensg = up2ensg.get(up, "")
    nxg.add_node(up, module=mod, gene_symbol=sym, ensembl=ensg)

for e in subg.es:
    src = subg.vs[e.source]["name"]
    tgt = subg.vs[e.target]["name"]
    t   = e["type"]
    nxg.add_edge(src, tgt, type=t)

# ── 4) Global color normalization across all bins ──────────────────────────
vals_all = []
for b in bins_to_plot:
    vals_all.extend(log2fc_pivot[b].values)
vals_all = np.array([v for v in vals_all if np.isfinite(v)])
absmax   = float(np.nanmax(np.abs(vals_all))) if vals_all.size else 1.0
absmax   = absmax if absmax > 0 else 1.0
norm     = Normalize(vmin=-absmax, vmax=absmax)
cmap     = coolwarm

# ── 5) Layout once for all panels ───────────────────────────────────────────
pos = nx.kamada_kawai_layout(nxg)

labels    = {n: d.get("gene_symbol") or n for n, d in nxg.nodes(data=True)}
act_edges = [(u, v) for u, v, d in nxg.edges(data=True) if d.get("type") == "activation"]
inh_edges = [(u, v) for u, v, d in nxg.edges(data=True) if d.get("type") == "inhibition"]

# ── 6) Plot one panel per bin ──────────────────────────────────────────────
ncols = len(bins_to_plot)
fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 8), squeeze=False)

for i, bin_name in enumerate(bins_to_plot):
    ax = axes[0, i]

    zvals = []
    for n, d in nxg.nodes(data=True):
        ensg = d.get("ensembl")
        if ensg in log2fc_pivot.index:
            val = log2fc_pivot.at[ensg, bin_name]
            if sig_mask:
                pval = padj_pivot.at[ensg, bin_name] if ensg in padj_pivot.index else np.nan
                if not (np.isfinite(pval) and pval < 0.05):
                    val = 0.0
            zvals.append(val)
            d["log2fc"] = val
        else:
            zvals.append(np.nan)
            d["log2fc"] = np.nan

    node_colors = [cmap(norm(v)) if np.isfinite(v) else (0.85,0.85,0.85,1.0) for v in zvals]

    nx.draw_networkx_nodes(nxg, pos, node_color=node_colors, node_size=320, alpha=0.95, ax=ax)
    nx.draw_networkx_labels(nxg, pos, labels=labels, font_size=7, font_color="black", ax=ax)
    if act_edges:
        nx.draw_networkx_edges(nxg, pos, edgelist=act_edges, edge_color="green",
                               arrowstyle="->", arrowsize=10, alpha=0.6,
                               connectionstyle='arc3,rad=0.2', width=1.5, ax=ax)
    if inh_edges:
        nx.draw_networkx_edges(nxg, pos, edgelist=inh_edges, edge_color="red",
                               arrowstyle="-|>", arrowsize=10, alpha=0.6,
                               connectionstyle='arc3,rad=0.2', width=1.5, ax=ax)

    ax.set_title(f"{bin_name} vs {baseline}", fontsize=11)
    ax.axis("off")

# shared colorbar
#sm = ScalarMappable(norm=norm, cmap=cmap)
#sm.set_array(vals_all)
#cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.7, pad=0.02)
#cbar.set_label("log2FC", rotation=270, labelpad=15)

fig.suptitle("Top-50 hubs — nodes colored by log2FC across bins", fontsize=16)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()


### color z-score

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable, coolwarm
import pandas as pd

# ── SETTINGS ────────────────────────────────────────────────────────────────
zscore_path  = "/storage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea/zscore_by_group.csv"
bins_to_plot = ["0_t_0.0000-0.5000", "1_t_3.0000-3.5000", "3_t_1.0000-1.5000", "2_t_3.5000-4.0000"]
top_n        = 100   # number of hubs by degree to include

# ── Load z-scores (index = Ensembl) ─────────────────────────────────────────
zscore_df = pd.read_csv(zscore_path, index_col=0)
missing = [b for b in bins_to_plot if b not in zscore_df.columns]
if missing:
    raise ValueError(f"Bins not in zscore_df: {missing}")
lowest_z = float(np.nanmin(zscore_df.values)) if zscore_df.size else 0.0
zscore_df = zscore_df.fillna(lowest_z)

# ── Pick top-N hubs (by total degree) and induce subgraph in igraph ─────────
deg   = g.degree()
topix = sorted(range(g.vcount()), key=lambda i: deg[i], reverse=True)[:top_n]
subg  = g.induced_subgraph(topix)

# ── Build NetworkX keyed by ENSEMBL (aligns with zscore_df) ─────────────────
nxg = nx.DiGraph()
for v in subg.vs:
    ensg = v["ensembl_gene_id"] if "ensembl_gene_id" in v.attributes() else ""
    if not ensg:
        continue
    nxg.add_node(
        ensg,
        gene_symbol=v["gene_symbol"] if "gene_symbol" in v.attributes() else "",
        uniprot_id=v["name"],   # keep UniProt for reference
    )

for e in subg.es:
    s = subg.vs[e.source]["ensembl_gene_id"] if "ensembl_gene_id" in subg.vs[e.source].attributes() else ""
    t = subg.vs[e.target]["ensembl_gene_id"] if "ensembl_gene_id" in subg.vs[e.target].attributes() else ""
    if s and t:
        nxg.add_edge(s, t, type=e["type"] if "type" in e.attributes() else "")

if nxg.number_of_nodes() == 0:
    raise ValueError("No nodes with 'ensembl_gene_id' found in the top-N subgraph.")

# ── Global normalization over this subgraph across all requested bins ───────
nodes = list(nxg.nodes())
Z = zscore_df.loc[zscore_df.index.intersection(nodes), bins_to_plot]
absmax = float(np.nanmax(np.abs(Z.values))) if Z.size else 1.0
absmax = absmax if absmax > 0 else 1.0
norm   = Normalize(vmin=-absmax, vmax=absmax)
cmap   = coolwarm

# ── Layout once (reuse across panels) ───────────────────────────────────────
pos = nx.kamada_kawai_layout(nxg)  # nice for dense hub networks

labels    = {n: (d.get("gene_symbol") or n) for n, d in nxg.nodes(data=True)}
act_edges = [(u, v) for u, v, d in nxg.edges(data=True) if d.get("type") == "activation"]
inh_edges = [(u, v) for u, v, d in nxg.edges(data=True) if d.get("type") == "inhibition"]
other_edges = [(u, v) for u, v, d in nxg.edges(data=True) if d.get("type") not in {"activation", "inhibition"}]

# ── Plot one panel per bin ──────────────────────────────────────────────────
ncols = len(bins_to_plot)
fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 8), squeeze=False)

for i, bin_name in enumerate(bins_to_plot):
    ax = axes[0, i]
    col = zscore_df[bin_name].reindex(nodes)  # align to network nodes
    vals = col.fillna(lowest_z).values

    node_colors = [cmap(norm(float(z))) for z in vals]

    nx.draw_networkx_nodes(nxg, pos, node_color=node_colors, node_size=320, alpha=0.95, ax=ax)
    nx.draw_networkx_labels(nxg, pos, labels=labels, font_size=7, font_color="black", ax=ax)

    if act_edges:
        nx.draw_networkx_edges(nxg, pos, edgelist=act_edges, edge_color="green",
                               arrowstyle="->", arrowsize=10, alpha=0.6,
                               connectionstyle='arc3,rad=0.2', width=1.5, ax=ax)
    if inh_edges:
        nx.draw_networkx_edges(nxg, pos, edgelist=inh_edges, edge_color="red",
                               arrowstyle="-|>", arrowsize=10, alpha=0.6,
                               connectionstyle='arc3,rad=0.2', width=1.5, ax=ax)
    if other_edges:
        nx.draw_networkx_edges(nxg, pos, edgelist=other_edges, edge_color="grey",
                               arrowstyle="->", arrowsize=10, alpha=0.4,
                               connectionstyle='arc3,rad=0.2', width=1.0, style="dotted", ax=ax)

    ax.set_title(f"{bin_name} (z-score)", fontsize=11)
    ax.axis("off")

# Shared colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array(Z.values.flatten() if Z.size else np.array([-1, 1]))
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.75, pad=0.02)
cbar.set_label("Z-score", rotation=270, labelpad=15)

fig.suptitle(f"Top-{top_n} hubs — nodes colored by z-score across bins", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## 9.2 Plot Z-core in the network plots

In [ ]:
groups_to_test = [
    "1_t_0.5000-1.5000", "1_t_1.5000-2.5000", "1_t_2.5000-3.0000", "1_t_3.0000-3.5000",
    "3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000",
    "2_t_1.0000-2.5000", "2_t_2.5000-3.0000", "2_t_3.0000-3.5000", "2_t_3.5000-4.0000"
]


### Plot a certain modules

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib import cm
import numpy as np

# --- tiny helpers for igraph attributes ---
def vget(v, key, default=None):
    try: return v[key]
    except KeyError: return default
def eget(e, key, default=None):
    try: return e[key]
    except KeyError: return default

# ─── Settings ────────────────────────────────────────────────────────────────
module_id   = 5
bin_to_plot = "2_t_3.5000-4.0000"
zscore_path = "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea/zscore_by_group.csv"

# ─── Load z-score data (index = Ensembl) ─────────────────────────────────────
zscore_df = pd.read_csv(zscore_path, index_col=0)
if bin_to_plot not in zscore_df.columns:
    raise ValueError(f"'{bin_to_plot}' not found in {zscore_df.columns.tolist()}")

# ─── igraph g → NetworkX (KEY BY ENSEMBL) ───────────────────────────────────
nxg = nx.DiGraph()
skipped = 0
for v in g.vs:
    ensg = vget(v, "ensembl_gene_id")      # <-- use Ensembl as node key
    if not ensg:
        skipped += 1
        continue
    nxg.add_node(
        ensg,
        module=vget(v, "module"),
        gene_symbol=vget(v, "gene_symbol", ""),
        uniprot_id=vget(v, "name", "")     # UniProt kept as attribute
    )
for e in g.es:
    src = vget(g.vs[e.source], "ensembl_gene_id")
    tgt = vget(g.vs[e.target], "ensembl_gene_id")
    if src and tgt:
        nxg.add_edge(src, tgt, type=eget(e, "type", ""))

if skipped:
    print(f"⚠️ Skipped {skipped} vertices without 'ensembl_gene_id'.")

# ─── Subgraph for selected module ────────────────────────────────────────────
members = [n for n, d in nxg.nodes(data=True) if d.get("module") == module_id]
if not members:
    raise ValueError(f"No nodes found for module {module_id}.")
subg = nxg.subgraph(members).copy()

# ─── Z-score coloring ────────────────────────────────────────────────────────
z_vals = zscore_df[bin_to_plot].reindex(subg.nodes())  # align by Ensembl
lowest_z = float(np.nanmin(zscore_df.values)) if zscore_df.size else 0.0
z_vals = z_vals.fillna(lowest_z)
absmax = float(max(abs(z_vals.min()), abs(z_vals.max()))) or 1.0
norm = Normalize(vmin=-absmax, vmax=absmax)
cmap = cm.coolwarm
node_colors = [cmap(norm(float(z))) for z in z_vals.values]

# ─── Layout ──────────────────────────────────────────────────────────────────
pos = nx.spring_layout(subg, seed=42, k=0.4)

# ─── Labels and Edge Types ───────────────────────────────────────────────────
labels = {n: d.get("gene_symbol", "") or n for n, d in subg.nodes(data=True)}
act_edges = [(u, v) for u, v, d in subg.edges(data=True) if d.get("type") == "activation"]
inh_edges = [(u, v) for u, v, d in subg.edges(data=True) if d.get("type") == "inhibition"]
other_edges = [(u, v) for u, v, d in subg.edges(data=True) if d.get("type") not in {"activation", "inhibition"}]

# ─── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))

nx.draw_networkx_nodes(subg, pos, node_color=node_colors, node_size=320, alpha=0.95, ax=ax)
if act_edges:
    nx.draw_networkx_edges(subg, pos, edgelist=act_edges, edge_color="green",
                           arrowstyle='->', arrowsize=8, alpha=0.7, width=1.5, ax=ax)
if inh_edges:
    nx.draw_networkx_edges(subg, pos, edgelist=inh_edges, edge_color="red",
                           arrowstyle='-|>', arrowsize=8, alpha=0.7, width=1.5, ax=ax)
if other_edges:
    nx.draw_networkx_edges(subg, pos, edgelist=other_edges, edge_color="grey",
                           arrowstyle='->', arrowsize=8, alpha=0.4, width=1.0, style="dotted", ax=ax)

nx.draw_networkx_labels(subg, pos, labels=labels, font_size=7, font_color="black",
                        bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.2"), ax=ax)

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array(z_vals.values)
cbar = plt.colorbar(sm, ax=ax, shrink=0.8); cbar.set_label("Z-score", rotation=270, labelpad=15)

ax.set_title(f"Module {module_id} — z-score in bin: {bin_to_plot}")
ax.axis("off")
plt.tight_layout()
plt.show()


### Plot a certain path (many modules)

In [ ]:
trajectories = {
    "Traj_1": ["0_t_0.0000-0.5000", "1_t_0.5000-1.5000", "1_t_1.5000-2.5000", "1_t_2.5000-3.0000", "1_t_3.0000-3.5000"],
    "Traj_2": ["0_t_0.0000-0.5000", "3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000"],
    "Traj_3": ["0_t_0.0000-0.5000", "3_t_0.0000-1.0000", "3_t_1.0000-1.5000", "3_t_1.5000-2.5000", "2_t_1.0000-2.5000", 
               "2_t_2.5000-3.0000", "2_t_3.0000-3.5000", "2_t_3.5000-4.0000"]
}

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib import cm
import numpy as np

# ── Helpers for safe igraph attribute access ─────────────────────────────────
def vget(v, key, default=None):
    try:
        return v[key]
    except KeyError:
        return default

def eget(e, key, default=None):
    try:
        return e[key]
    except KeyError:
        return default

# ── Settings ────────────────────────────────────────────────────────────────
module_id = 5
bins_to_plot = [
    "0_t_0.0000-0.5000",  "1_t_3.0000-3.5000", "3_t_1.0000-1.5000", "2_t_3.5000-4.0000"
]
zscore_path = "/storage/users/job37yv/Projects/PANC_cancer/analysis/network_module_gsea/zscore_by_group.csv"

# ── Load z-scores (index = Ensembl) ─────────────────────────────────────────
zscore_df = pd.read_csv(zscore_path, index_col=0)
for col in bins_to_plot:
    if col not in zscore_df.columns:
        raise ValueError(f"'{col}' not in zscore_df columns.")
lowest_z = np.nanmin(zscore_df.values) if zscore_df.size else 0.0
zscore_df = zscore_df.fillna(lowest_z)

# ── Build NetworkX keyed by ENSEMBL ─────────────────────────────────────────
nxg = nx.DiGraph()
missing_ensg = 0
for v in g.vs:
    ensg = vget(v, "ensembl_gene_id")
    if not ensg:
        missing_ensg += 1
        continue
    nxg.add_node(
        ensg,
        module=vget(v, "module"),
        gene_symbol=vget(v, "gene_symbol", ""),
        uniprot_id=vget(v, "name", "")  # UniProt kept as attribute
    )

for e in g.es:
    src = vget(g.vs[e.source], "ensembl_gene_id")
    tgt = vget(g.vs[e.target], "ensembl_gene_id")
    if src and tgt:
        nxg.add_edge(src, tgt, type=eget(e, "type", ""))

if missing_ensg:
    print(f"⚠️ Skipped {missing_ensg} vertices without 'ensembl_gene_id'.")

# ── Subgraph for the module ─────────────────────────────────────────────────
members = [n for n, d in nxg.nodes(data=True) if d.get("module") == module_id]
if not members:
    raise ValueError(f"No nodes found for module {module_id}.")
subg = nxg.subgraph(members).copy()

# Ensure all subgraph nodes are present in zscore_df index
not_in_z = [n for n in subg.nodes() if n not in zscore_df.index]
if not_in_z:
    print(f"⚠️ {len(not_in_z)} module nodes missing from zscore_df index (showing up to 10): {not_in_z[:10]}")

# ── Global normalization over this subgraph across all bins ─────────────────
Z = zscore_df.loc[zscore_df.index.intersection(subg.nodes()), bins_to_plot]
absmax = float(np.nanmax(np.abs(Z.values))) if Z.size else 1.0
absmax = absmax if absmax > 0 else 1.0
norm = Normalize(vmin=-absmax, vmax=absmax)
cmap = cm.coolwarm

# ── Layout once (reuse across panels) ───────────────────────────────────────
pos = nx.spring_layout(subg, seed=42, k=0.4)

# ── Subplots (vertical) ─────────────────────────────────────────────────────
fig, axes = plt.subplots(len(bins_to_plot), 1, figsize=(10, 8 * len(bins_to_plot)), squeeze=False)

labels = {n: d.get("gene_symbol") or n for n, d in subg.nodes(data=True)}
act_edges = [(u, v) for u, v, d in subg.edges(data=True) if d.get("type") == "activation"]
inh_edges = [(u, v) for u, v, d in subg.edges(data=True) if d.get("type") == "inhibition"]
other_edges = [(u, v) for u, v, d in subg.edges(data=True) if d.get("type") not in {"activation", "inhibition"}]

for i, bin_name in enumerate(bins_to_plot):
    ax = axes[i, 0]
    z_vals = zscore_df[bin_name].reindex(subg.nodes()).fillna(lowest_z)
    node_colors = [cmap(norm(float(z))) for z in z_vals.values]

    nx.draw_networkx_nodes(subg, pos, node_color=node_colors, node_size=400, alpha=0.95, ax=ax)
    if act_edges:
        nx.draw_networkx_edges(subg, pos, edgelist=act_edges, edge_color="green",
                               arrowstyle='->', arrowsize=10, alpha=0.7, ax=ax, width=1.5)
    if inh_edges:
        nx.draw_networkx_edges(subg, pos, edgelist=inh_edges, edge_color="red",
                               arrowstyle='-|>', arrowsize=10, alpha=0.7, ax=ax, width=1.5)
    if other_edges:
        nx.draw_networkx_edges(subg, pos, edgelist=other_edges, edge_color="grey",
                               arrowstyle='->', arrowsize=10, alpha=0.4, ax=ax, width=1.0, style="dotted")

    nx.draw_networkx_labels(subg, pos, labels=labels, font_size=8, font_color="black",
                            bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.2"), ax=ax)

    ax.set_title(bin_name, fontsize=11)
    ax.axis("off")

# Shared colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array(Z.values.flatten() if Z.size else np.array([-1, 1]))
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.1, pad=0.01)
cbar.set_label("Z-score", rotation=270, labelpad=15)

fig.suptitle(f"Module {module_id} — Z-scores across trajectory bins", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 9.3 Plot logFC in the network plots

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib import cm
import numpy as np

def vget(v, key, default=None):
    try: return v[key]
    except KeyError: return default
def eget(e, key, default=None):
    try: return e[key]
    except KeyError: return default

# ---- Settings ----
module_id    = 5
baseline     = "0_t_0.0000-0.5000"   # your DE reference
# If you want to pick specific bins, put them here (baseline will be removed automatically)
requested_bins = ["0_t_0.0000-0.5000", "1_t_3.0000-3.5000", "3_t_1.0000-1.5000", "2_t_3.5000-4.0000"]
sig_mask     = False  # set True to grey out padj >= 0.05

# ---- Build wide log2FC matrix (genes × bins) for the chosen baseline ----
de_base = final_de_df[final_de_df["baseline"] == baseline].copy()

# auto-derive bins available for this baseline
available_bins = sorted(de_base["bin"].unique().tolist())

# use requested bins if given; otherwise use all available
if requested_bins:
    # drop baseline if present and keep only bins that exist
    bins_to_plot = [b for b in requested_bins if (b != baseline and b in available_bins)]
else:
    bins_to_plot = [b for b in available_bins if b != baseline]

if not bins_to_plot:
    raise ValueError(f"No bins to plot: after removing baseline '{baseline}', none remain from {requested_bins or available_bins}.")

# pivots
log2fc_pivot = de_base.pivot(index="ensembl_gene_id", columns="bin", values="log2fc")
padj_pivot   = de_base.pivot(index="ensembl_gene_id", columns="bin", values="padj") if sig_mask else None

# ---- igraph g -> NetworkX (keyed by Ensembl) ----
nxg = nx.DiGraph()
for v in g.vs:
    ensg = vget(v, "ensembl_gene_id")
    if not ensg: 
        continue
    nxg.add_node(ensg,
                 module=vget(v, "module"),
                 gene_symbol=vget(v, "gene_symbol", ""),
                 uniprot_id=vget(v, "name", ""))

for e in g.es:
    s = vget(g.vs[e.source], "ensembl_gene_id"); t = vget(g.vs[e.target], "ensembl_gene_id")
    if s and t:
        nxg.add_edge(s, t, type=eget(e, "type", ""))

# subgraph by module
members = [n for n, d in nxg.nodes(data=True) if d.get("module") == module_id]
if not members:
    raise ValueError(f"No nodes found for module {module_id}.")
subg = nxg.subgraph(members).copy()

# align matrix to subgraph nodes
genes = list(subg.nodes())
L = log2fc_pivot.reindex(index=genes, columns=bins_to_plot)

if L.size == 0:
    raise ValueError("No overlap between module genes and DE results for these bins.")

# global symmetric normalization across all panels
absmax = float(np.nanmax(np.abs(L.values))) if np.isfinite(L.values).any() else 1.0
absmax = absmax if absmax > 0 else 1.0
norm = Normalize(vmin=-absmax, vmax=absmax)
cmap = cm.coolwarm

# optional significance mask
if sig_mask:
    P = padj_pivot.reindex(index=genes, columns=bins_to_plot)
    sig_mask_mat = (P.values < 0.05)
else:
    sig_mask_mat = np.ones_like(L.values, dtype=bool)

# layout and plot
pos = nx.spring_layout(subg, seed=42, k=0.4)
fig, axes = plt.subplots(len(bins_to_plot), 1, figsize=(10, 8 * len(bins_to_plot)), squeeze=False)

labels = {n: d.get("gene_symbol") or n for n, d in subg.nodes(data=True)}
act_edges = [(u, v) for u, v, d in subg.edges(data=True) if d.get("type") == "activation"]
inh_edges = [(u, v) for u, v, d in subg.edges(data=True) if d.get("type") == "inhibition"]
other_edges = [(u, v) for u, v, d in subg.edges(data=True) if d.get("type") not in {"activation", "inhibition"}]

for i, bin_name in enumerate(bins_to_plot):
    ax = axes[i, 0]
    col = L[bin_name]
    vals = col.values.copy()
    if sig_mask:
        vals[~sig_mask_mat[:, i]] = 0.0  # grey/white non-significant

    node_colors = [cmap(norm(float(z))) for z in vals]
    nx.draw_networkx_nodes(subg, pos, node_color=node_colors, node_size=400, alpha=0.95, ax=ax)
    if act_edges:
        nx.draw_networkx_edges(subg, pos, edgelist=act_edges, edge_color="green", arrowstyle='->', arrowsize=10, alpha=0.7, ax=ax, width=1.5)
    if inh_edges:
        nx.draw_networkx_edges(subg, pos, edgelist=inh_edges, edge_color="red", arrowstyle='-|>', arrowsize=10, alpha=0.7, ax=ax, width=1.5)
    if other_edges:
        nx.draw_networkx_edges(subg, pos, edgelist=other_edges, edge_color="grey", arrowstyle='->', arrowsize=10, alpha=0.4, ax=ax, width=1.0, style="dotted")

    nx.draw_networkx_labels(subg, pos, labels=labels, font_size=8, font_color="black",
                            bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.2"), ax=ax)

    ax.set_title(f"{bin_name} vs {baseline} (log2FC)", fontsize=11)
    ax.axis("off")

# shared colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array(L.values.flatten())
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.1, pad=0.01)
cbar.set_label("log2FC", rotation=270, labelpad=15)

fig.suptitle(f"Module {module_id} — log2FC across bins (baseline = {baseline})", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 9.4 Viusalize full network

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 1) Convert igraph `g` to a NetworkX DiGraph carrying the "type" attribute
nxg = nx.DiGraph()
for v in g.vs:
    nxg.add_node(v["name"])
for e in g.es:
    src = g.vs[e.tuple[0]]["name"]
    tgt = g.vs[e.tuple[1]]["name"]
    nxg.add_edge(src, tgt, type=e["type"])

# 2) Compute a layout (spring layout for the whole graph)
pos = nx.spring_layout(nxg, seed=42, k=0.15, iterations=50)

# 3) Split edges by type
act_edges = [(u, v) for u, v, d in nxg.edges(data=True) if d["type"] == "activation"]
inh_edges = [(u, v) for u, v, d in nxg.edges(data=True) if d["type"] == "inhibition"]

# 4) Plot
plt.figure(figsize=(12,12))

# draw nodes
nx.draw_networkx_nodes(
    nxg, pos,
    node_size=20,
    node_color="lightgray",
    alpha=0.7
)

# draw activation edges
nx.draw_networkx_edges(
    nxg, pos,
    edgelist=act_edges,
    edge_color="green",
    arrowsize=6,
    alpha=0.5,
    label="activation"
)

# draw inhibition edges
nx.draw_networkx_edges(
    nxg, pos,
    edgelist=inh_edges,
    edge_color="red",
    arrowsize=6,
    alpha=0.5,
    label="inhibition"
)

# optional: no labels for clarity
plt.axis("off")
plt.legend(loc="upper right")
plt.title("Full network: activation (green) vs inhibition (red)")
plt.tight_layout()
plt.show()


# 10.  Export netwrok for Cytoscape / yEd

## 10.0 Save graph as python graphml (done before already)

In [ ]:
import pandas as pd
import igraph as ig

# ----------------------------
# 1. Inspect attributes
# ----------------------------
print("📌 Vertex attributes:", g.vs.attributes())
print("📌 Edge attributes:", g.es.attributes())
print(f"🔢 Vertices: {g.vcount()}, Edges: {g.ecount()}")

# ----------------------------
# 2. Vertex attributes DataFrame
# ----------------------------
vertex_df = pd.DataFrame({attr: g.vs[attr] for attr in g.vs.attributes()})
print("\n🧬 Vertex table (all attributes):")
print(vertex_df.head(10))   # preview first 10

# ----------------------------
# 3. Edge attributes DataFrame
# ----------------------------
edge_attr_dict = {attr: g.es[attr] for attr in g.es.attributes()}
edge_df = pd.DataFrame(edge_attr_dict)

# Add endpoints
src = [e.tuple[0] for e in g.es]
tgt = [e.tuple[1] for e in g.es]
edge_df.insert(0, "source_idx", src)
edge_df.insert(1, "target_idx", tgt)

# Map to Ensembl IDs (or names)
if "name" in g.vs.attributes():
    idx2name = dict(enumerate(g.vs["name"]))
    edge_df["source_name"] = [idx2name[i] for i in src]
    edge_df["target_name"] = [idx2name[i] for i in tgt]

print("\n🕸️ Edge table (all attributes):")
print(edge_df.head(10))   # preview first 10

# ----------------------------
# 4. Save graph with attributes
# ----------------------------
g.write_graphml("/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_with_attributes.graphml")
print("\n💾 Saved graph as 'graph_with_attributes.graphml' (keeps all attributes).")

# Optional: also save CSVs for inspection
vertex_df.to_csv("/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_vertices.csv", index=False)
edge_df.to_csv("/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/graph_edges.csv", index=False)
print("💾 Also saved 'graph_vertices.csv' and 'graph_edges.csv'")


## 10.1 Cytoscape

In [ ]:
import pandas as pd

# --- 1) Export network as .sif ---------------------------------------------
# Cytoscape SIF format: source <interaction-type> target
sif_path = "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/network.sif"

with open(sif_path, "w") as f:
    for e in g.es:
        src = g.vs[e.source]["ensembl_gene_id"]
        tgt = g.vs[e.target]["ensembl_gene_id"]
        inter = e["type"] if "type" in g.es.attributes() else "interacts_with"
        if src and tgt:  # only export if Ensembl IDs exist
            f.write(f"{src}\t{inter}\t{tgt}\n")

print(f"💾 SIF network written: {sif_path}")

# --- 2) Node attributes -----------------------------------------------------
# expr_mean2 already has columns: ensembl_gene_id, UniProtID, gene_symbol, cluster_id, ...
node_attr_path = "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/node_attributes.tsv"
expr_mean2.to_csv(node_attr_path, sep="\t", index=False)
print(f"💾 Node attributes written: {node_attr_path}")

# --- 3) Edge attributes -----------------------------------------------------
edge_attr_path = "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/edge_attributes.tsv"
edge_attr_dict = {attr: g.es[attr] for attr in g.es.attributes()}
edge_df = pd.DataFrame(edge_attr_dict)

# add source/target Ensembl IDs
edge_df["source"] = [g.vs[e.source]["ensembl_gene_id"] for e in g.es]
edge_df["target"] = [g.vs[e.target]["ensembl_gene_id"] for e in g.es]

edge_df.to_csv(edge_attr_path, sep="\t", index=False)
print(f"💾 Edge attributes written: {edge_attr_path}")


## 10.2 Gramphml (didnt really work)

In [ ]:
from itertools import islice

# ---- Vertices ----
print("Vertices and their attributes (head):")
for v in islice(g.vs, 5):   # first 5 vertices
    print(f"Vertex {v.index}:")
    for attr in g.vs.attributes():
        print(f"  {attr}: {v[attr]}")
    print()

# ---- Edges ----
print("Edges and their attributes (head):")
for e in islice(g.es, 5):   # first 5 edges
    print(f"Edge {e.index}: {e.source} -> {e.target}")
    for attr in g.es.attributes():
        print(f"  {attr}: {e[attr]}")
    print()


In [ ]:
# pip install igraph
import xml.etree.ElementTree as ET

def export_g_to_yed_graphml(g, path, use_positions=False, width=30.0, height=30.0):
    """
    Export existing igraph.Graph 'g' to yEd GraphML with:
      - node ids n0, n1, ...
      - NodeLabel from 'gene_symbol' (fallback: 'name' -> 'ensembl_gene_id' -> index)
      - Edge 'type' mapped: activation -> green arrow; inhibition/inhibtion -> red T-bar.
      - If use_positions=True and vertices have x,y, Geometry is set; else yEd will auto-layout.
    """
    # namespaces
    ns = {
        'graphml': "http://graphml.graphdrawing.org/xmlns",
        'y': "http://www.yworks.com/xml/graphml",
        'yed': "http://www.yworks.com/xml/yed/3",
        'xsi': "http://www.w3.org/2001/XMLSchema-instance",
        'java': "http://www.yworks.com/xml/yfiles-common/1.0/java",
        'sys': "http://www.yworks.com/xml/yfiles-common/markup/primitives/2.0",
        'x': "http://www.yworks.com/xml/yfiles-common/markup/2.0",
    }
    for prefix, uri in ns.items():
        ET.register_namespace(prefix if prefix != 'graphml' else '', uri)

    root = ET.Element(
        "{" + ns['graphml'] + "}graphml",
        {"{" + ns['xsi'] + "}schemaLocation":
         f"{ns['graphml']} http://www.yworks.com/xml/schema/graphml/1.1/ygraphml.xsd"}
    )

    def add_key(_id, _for=None, attr_name=None, attr_type=None, yfiles_type=None):
        attrs = {"id": _id}
        if _for:        attrs["for"] = _for
        if attr_name:   attrs["attr.name"] = attr_name
        if attr_type:   attrs["attr.type"] = attr_type
        if yfiles_type: attrs["yfiles.type"] = yfiles_type
        ET.SubElement(root, "key", attrs)

    # keys like your template
    add_key("d0", "graph",  "Description", "string")
    add_key("d1", "port",   yfiles_type="portgraphics")
    add_key("d2", "port",   yfiles_type="portgeometry")
    add_key("d3", "port",   yfiles_type="portuserdata")
    add_key("d4", "node",   "url", "string")
    add_key("d5", "node",   "description", "string")
    add_key("d6", "node",   yfiles_type="nodegraphics")
    add_key("d7", "graphml", yfiles_type="resources")
    add_key("d8", "edge",   "url", "string")
    add_key("d9", "edge",   "description", "string")
    add_key("d10","edge",   yfiles_type="edgegraphics")

    graph_el = ET.SubElement(root, "graph", {
        "id": "G",
        "edgedefault": "directed" if g.is_directed() else "undirected"
    })
    ET.SubElement(graph_el, "data", {"key": "d0"})  # empty graph description

    v_attrs = set(g.vs.attributes())
    e_attrs = set(g.es.attributes())

    def node_label(v):
        for attr in ("gene_symbol", "name", "ensembl_gene_id"):
            if attr in v_attrs:
                val = v[attr]
                if val not in (None, ""):
                    return str(val)
        return str(v.index)

    def edge_type(e):
        val = None
        if "type" in e_attrs:
            val = e["type"]
        elif "sign" in e_attrs:
            try:
                val = "activation" if float(e["sign"]) > 0 else "inhibition"
            except Exception:
                val = None
        s = (str(val).strip().lower() if val is not None else "")
        if s in ("inhibition", "inhibtion"):   # tolerate typo
            return "inhibition"
        return "activation"

    def style_for(etype):
        if etype == "inhibition":
            return "#FF0000", "t_shape"
        return "#99CC00", "standard"

    use_xy = use_positions and ("x" in v_attrs) and ("y" in v_attrs)

    # nodes
    for v in g.vs:
        nid = f"n{v.index}"
        node_el = ET.SubElement(graph_el, "node", {"id": nid})
        ET.SubElement(node_el, "data", {"key": "d5"})  # empty node description

        d6 = ET.SubElement(node_el, "data", {"key": "d6"})
        shape = ET.SubElement(d6, f"{{{ns['y']}}}ShapeNode")

        if use_xy:
            try:
                x = float(v["x"]) if v["x"] is not None else 0.0
                y = float(v["y"]) if v["y"] is not None else 0.0
            except Exception:
                x, y = 0.0, 0.0
        else:
            x, y = 0.0, 0.0

        ET.SubElement(shape, f"{{{ns['y']}}}Geometry",
                      {"x": f"{x}", "y": f"{y}", "width": f"{float(width)}", "height": f"{float(height)}"})
        ET.SubElement(shape, f"{{{ns['y']}}}Fill", {"color":"#FFCC00", "transparent":"false"})
        ET.SubElement(shape, f"{{{ns['y']}}}BorderStyle",
                      {"color":"#000000", "type":"line", "width":"1.0", "raised":"false"})

        nlabel = ET.SubElement(shape, f"{{{ns['y']}}}NodeLabel",
                               {"alignment":"center",
                                "autoSizePolicy":"content",
                                "fontFamily":"Dialog",
                                "fontSize":"12",
                                "fontStyle":"plain",
                                "hasBackgroundColor":"false",
                                "hasLineColor":"false",
                                "visible":"true",
                                "textColor":"#000000",
                                "modelName":"custom",
                                "horizontalTextPosition":"center",
                                "verticalTextPosition":"bottom",
                                "x":"0.0","y":"0.0","width":"40.0","height":"20.0",
                                "iconTextGap":"4"})
        nlabel.text = node_label(v)
        lm = ET.SubElement(nlabel, f"{{{ns['y']}}}LabelModel")
        ET.SubElement(lm, f"{{{ns['y']}}}SmartNodeLabelModel", {"distance":"4.0"})
        mp = ET.SubElement(nlabel, f"{{{ns['y']}}}ModelParameter")
        ET.SubElement(mp, f"{{{ns['y']}}}SmartNodeLabelModelParameter",
                      {"labelRatioX":"0.0","labelRatioY":"0.0",
                       "nodeRatioX":"0.0","nodeRatioY":"0.0",
                       "offsetX":"0.0","offsetY":"0.0","upX":"0.0","upY":"-1.0"})
        ET.SubElement(shape, f"{{{ns['y']}}}Shape", {"type":"rectangle"})

    # edges
    for e in g.es:
        eid = f"e{e.index}"
        src = f"n{e.source}"
        tgt = f"n{e.target}"
        etype = edge_type(e)
        color, arrow = style_for(etype)

        edge_el = ET.SubElement(graph_el, "edge", {"id": eid, "source": src, "target": tgt})
        ET.SubElement(edge_el, "data", {"key": "d9"})  # empty edge description (like template)

        d10 = ET.SubElement(edge_el, "data", {"key": "d10"})
        poly = ET.SubElement(d10, f"{{{ns['y']}}}PolyLineEdge")
        ET.SubElement(poly, f"{{{ns['y']}}}Path", {"sx":"0.0","sy":"0.0","tx":"0.0","ty":"0.0"})
        ET.SubElement(poly, f"{{{ns['y']}}}LineStyle", {"color": color, "type":"line", "width":"1.0"})
        ET.SubElement(poly, f"{{{ns['y']}}}Arrows", {"source":"none", "target": arrow})
        ET.SubElement(poly, f"{{{ns['y']}}}BendStyle", {"smoothed":"false"})

    # resources bucket
    data_res = ET.SubElement(root, "data", {"key": "d7"})
    ET.SubElement(data_res, f"{{{ns['y']}}}Resources")

    ET.ElementTree(root).write(path, encoding="UTF-8", xml_declaration=True)
    print(f"💾 GraphML written: {path}")


In [ ]:
export_g_to_yed_graphml(g, "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/network_yed.graphml", use_positions=False)

## 10.3 save all sub networks (doesnt work)

In [ ]:
import os
import pandas as pd
import igraph as ig

outdir = "/bigstorage/users/job37yv/Projects/PANC_cancer/analysis/omnipath/subgraphs"
os.makedirs(outdir, exist_ok=True)

for group, g_sub in group_graphs.items():
    print(f"\n=== Saving subgraph for {group} ===")
    print(f"Nodes: {g_sub.vcount()}, Edges: {g_sub.ecount()}")

    # --- Vertex table ---
    vertex_df = pd.DataFrame({attr: g_sub.vs[attr] for attr in g_sub.vs.attributes()})
    print(vertex_df.head(5))
    
    # --- Edge table ---
    edge_attr_dict = {attr: g_sub.es[attr] for attr in g_sub.es.attributes()}
    edge_df = pd.DataFrame(edge_attr_dict)

    src = [e.tuple[0] for e in g_sub.es]
    tgt = [e.tuple[1] for e in g_sub.es]
    edge_df.insert(0, "source_idx", src)
    edge_df.insert(1, "target_idx", tgt)

    if "name" in g_sub.vs.attributes():
        idx2name = dict(enumerate(g_sub.vs["name"]))
        edge_df["source_name"] = [idx2name[i] for i in src]
        edge_df["target_name"] = [idx2name[i] for i in tgt]

    print(edge_df.head(5))

    # --- Save files ---
    g_sub.write_graphml(os.path.join(outdir, f"{group}.graphml"))
    vertex_df.to_csv(os.path.join(outdir, f"{group}_nodes.csv"), index=False)
    edge_df.to_csv(os.path.join(outdir, f"{group}_edges.csv"), index=False)

    print(f"💾 Saved {group}.graphml, {group}_nodes.csv, {group}_edges.csv")
